# NB03 - Modeling Panel Construction

---

## Purpose

Constructs the **loan-month level modeling panel** consumed by all downstream notebooks.

The notebook prepares a supervised credit-risk panel for later delinquency-risk prediction and classifier benchmarking (Lessmann et al., 2015; Fitzpatrick & Mues, 2016).

Each row represents a single loan at a single observation month `t` and carries:
- loan status and dynamic features **at** month `t`;
- pre-aggregated lagged delinquency history from the 12-month window **before** `t`;
- binary target `y = 1` if the loan reaches **60+ DPD or REO Acquisition** in `(t, t+12]`;
- static origination characteristics cleaned to their documented disclosure semantics.

This creates a loan-month risk panel: covariates are measured at or before the observation month, while the binary event indicator is measured over a fixed future 12-month window. The general organisation - repeated binary observations per unit with time-varying covariates - is consistent with discrete-time event-history data structures (Singer & Willett, 1993; Shumway, 2001; Wooldridge, 2010). The 12-month cumulative label horizon differs from the single-period hazard of canonical grouped-duration models; this design is better described as a fixed-horizon binary prediction panel. The strict separation between information observable at `t` and outcomes realized after `t` is also a leakage-prevention design: features are only legitimate if they would be available at the prediction time, while target information from the forward window is excluded from the feature set (Kaufman et al., 2012).

**Model scope:** The base population filter (`zero_balance_code IS NULL AND status IN ('0','00','1')`) means the panel models **transition from current or 30-DPD into 60+ DPD or REO Acquisition over the next 12 months**. Loans already at 60+ DPD at observation time are excluded from the base population.

**Dataset scope:** As defined in NB01.

---

## Panel Design Decisions

| Decision | Choice | Rationale |
|---|---|---|
| **Target** | `y = 1` if 60+ DPD **or** REO Acquisition in `(t, t+12]` | Uses Freddie Mac monthly-performance status codes: numeric delinquency values `2+` capture 60+ DPD, while `RA` and legacy `R` capture REO Acquisition status (Freddie Mac, 2026b, 2026c). |
| **Base population** | Active, current or 30-DPD, `loan_age >= 1` | Transition-risk model; excludes loans already in serious delinquency at observation time. |
| **`loan_age = 0` exclusion** | Modeling decision, not data cleaning | `loan_age = 0` can occur in valid SFLLD records; the analysis mechanically requires reported `loan_age >= 1`. |
| **Target horizon** | 12-month forward | Chosen as a fixed one-year forward transition-risk horizon. The horizon is a research-design choice, not an SFLLD field definition. |
| **Feature timing / leakage boundary** | Features observed at or before `t`; target observed in `(t, t+12]` | Implements learn-predict separation: the model is allowed to use only information that is legitimate at the observation month, not information revealed during or after the target window (Kaufman et al., 2012). |
| **Payoff / maturity** | Code-01 termination yields `y = 0` unless a target event occurs first | Treated as a competing termination risk; NB06 tests robustness by excluding all rows with code 01 inside the horizon (Deng et al., 2000). |
| **`modification_flag`** | `IN ('Y','P')` | `Y` indicates current-period modification and `P` indicates prior-period modification (Freddie Mac, 2026b). |
| **`payment_deferral_flag`** | `IN ('Y','P')` | Retained for diagnostics but excluded from active model features because payment deferral is a later-regime policy/workout feature. |
| **FICO/LTV/CLTV missing** | **No active imputation** - preserve `NULL` + missingness flags | Disclosure sentinels and regime-invalid values are kept as `NULL`; explicit missingness flags carry structural information downstream. This avoids treating disclosure-driven missingness as ordinary numerical data. The Freddie Mac documentation defines the disclosure sentinels, while Rubin (1976) and Little & Rubin (2019) provide the missing-data framework motivating explicit treatment of the missingness mechanism rather than uncritical numeric imputation.|
| **DTI missing** | **No imputation** - preserve `NULL` + missingness flags only | `DTI = 999` conflates `>65%`, genuinely unavailable values, and Relief Refinance masking. Imputing a numeric DTI would fabricate underwriting information (Freddie Mac, 2026a, 2026b). |
| **CLTV valid range** | pre-2018Q2 non-relief: 6-200; later / Relief Refi: 1-998 | CLTV upper bound is 200 before 2018Q2 for non-Relief Refinance loans; the 105 upper bound applies to LTV, not CLTV (Freddie Mac, 2026a, 2026b). |
| **HARP flag** | Relief Refi + original LTV > 80 + vintage 2009-2018 | Matches the FAQ definition of HARP population in the SFLLD (Freddie Mac, 2026a). |
| **Channel** | Retained but hard-excluded | Freddie Mac did not collect granular broker/correspondent/retail channel information before 2008, so pre-2008 `T` values are not a stable economic channel signal (Freddie Mac, 2026a). |
| **Current LTV / ELTV** | ELTV excluded from active features | ELTV is available only from April 2017 onward and is therefore absent in the 1999-2003 fitted-training window (Freddie Mac, 2026a, 2026c). Excluding it preserves the frozen-model design, but leaves an important negative-equity/current-LTV channel unobserved (Campbell & Cocco, 2015; Foote et al., 2008). |
| **Splits** | Time-based, not random | Temporal splitting avoids random mixing across macro regimes and is consistent with blocked-validation guidance for dependent temporal data (Roberts et al., 2017). |
| **CP caveat** | Exchangeability not assumed | Standard conformal validity relies on exchangeability; calendar-time regime shifts break that benchmark, while repeated loan-month observations add structured dependence. Downstream notebooks therefore audit coverage empirically (Vovk et al., 2005; Foygel Barber et al., 2023). |
| **first_time_homebuyer** | `NULL`/blank → `'Not_Applicable'`, code `9` → `'Unknown'` | Interpretation based on FAQ Q38 context (blanks appear for Investment Properties, Second Homes, and Refinance transactions). Note: the current User Guide and Release Notes define `9 = Not Available or Not Applicable` as a combined code without a separate blank entry. The blank=Not_Applicable / 9=Unknown separation is an inference from FAQ Q38, not a clean data-dictionary specification. |
| **in_workout_plan_flag** | `borrower_assistance_status_code IS NOT NULL` | Captures F/R/T-type workout states; retained as metadata, not model feature |
| **deferred_upb_ratio** | `current_non_interest_bearing_upb / current_actual_upb` | Near-constant in the early training window; validation checks explicitly test availability |
| **Single frozen model** | Train on early-history window; downstream CP evaluated on later regimes | Supports regime-based out-of-time evaluation in later notebooks |

## Notebook Structure

| Section | Description |
|---|---|
| **§0 Setup & Configuration** | Environment, imports, paths, DuckDB views over the NB01 Parquet outputs, ingestion-manifest verification, and the canonical split windows, target condition, and feature-eligibility rules. |
| **§1 Lookup Tables** | State→Census-division map and the `origination_semantics` view (`true_vintage_year`, `true_vintage_quarter`, `origination_rule_group`). |
| **§2 Base Population Diagnostics** | Eligible loan-month counts by year and split under the base-population rule. |
| **§3 Precomputed Lookup Tables** | `precomp_orig`, `precomp_label`, `precomp_prepay`, `precomp_lagged`, a right-truncation check, and the year-by-year panel assembly. |
| **§4 Post-Processing** | Combines year files into the split-partitioned panel and adds subgroup variables for evaluation and Mondrian CP. |
| **§5 Validation Checks** | Ten-check QC battery (enumerated at the section header). |
| **§6 Output Manifest** | Writes `panel_manifest.json`, the contract with downstream notebooks. |
| **§7 Close Connection** | Closes DuckDB. |

## Output of This Notebook

This notebook is the core data-to-modeling interface of the pipeline: it converts the cleaned Freddie Mac Parquet data into a temporally structured, semantically documented, and validation-checked loan-month panel, and writes `panel_manifest.json` for downstream reproducibility.

---
## Section 0 · Setup & Configuration

### 0.1 · Install Dependencies

In [ ]:
%pip install -q duckdb pyarrow pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 198.6 MB/s eta 0:00:00


### 0.2 · Imports

In [ ]:
from __future__ import annotations
import json, os, shutil, sys, time
from datetime import datetime, timezone
from pathlib import Path

import duckdb
import pandas as pd
from tqdm.auto import tqdm

IS_COLAB = "google.colab" in sys.modules or Path("/content").exists()
print(f"Running on Colab : {IS_COLAB}")
print(f"Python           : {sys.version.split()[0]}")
print(f"DuckDB           : {duckdb.__version__}")

Running on Colab : True
Python           : 3.12.13
DuckDB           : 1.5.4


### 0.3 · Drive Mount & Path Configuration

Source data is read from Drive.
All outputs are written to local NVMe during computation, then synced to Drive.


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive/master_thesis")

# ── Source Parquet (read from Drive) ──────────────────────────────────────────
PARQUET_ROOT              = DRIVE_ROOT / "data" / "parquet"
ORIGINATION_OUT           = PARQUET_ROOT / "origination"
PERFORMANCE_OUT           = PARQUET_ROOT / "performance"
PERFORMANCE_BY_PERIOD_OUT = PARQUET_ROOT / "performance_by_period"

orig_glob        = (ORIGINATION_OUT           / "**" / "*.parquet").as_posix()
perf_glob        = (PERFORMANCE_OUT           / "**" / "*.parquet").as_posix()
perf_period_glob = (PERFORMANCE_BY_PERIOD_OUT / "**" / "*.parquet").as_posix()

# ── Local NVMe output dirs ──────────────────────────────────────────────────────
LOCAL_ROOT      = Path("/content") if IS_COLAB else DRIVE_ROOT
PRECOMP_DIR     = LOCAL_ROOT / "precomp"
PANEL_TEMP_DIR  = LOCAL_ROOT / "panel_temp"
PANEL_FINAL_DIR = LOCAL_ROOT / "panel_final"

for d in [PRECOMP_DIR, PANEL_TEMP_DIR, PANEL_FINAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Drive destination dirs ────────────────────────────────────────────────────
DRIVE_PRECOMP = DRIVE_ROOT / "data" / "precomp"
DRIVE_PANEL   = DRIVE_ROOT / "data" / "panel"
MANIFEST_DIR = DRIVE_ROOT / "manifests"
for d in [DRIVE_PRECOMP, DRIVE_PANEL, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── DuckDB temp spill (local NVMe) ───────────────────────────────
DUCKDB_TEMP = str(LOCAL_ROOT / "duckdb_tmp")
Path(DUCKDB_TEMP).mkdir(parents=True, exist_ok=True)

for p, name in [(ORIGINATION_OUT,"origination"), (PERFORMANCE_OUT,"performance (vintage)"),
                (PERFORMANCE_BY_PERIOD_OUT,"performance (by period)")]:
    status = "✓" if p.exists() else "✗ MISSING - run 01_data_preparation.ipynb"
    print(f"  {status}  {name}: {p}")

print(f"\n  Local precomp : {PRECOMP_DIR}")
print(f"  Local panel   : {PANEL_FINAL_DIR}")
print(f"  Drive panel   : {DRIVE_PANEL}")

def sync_dir_to_drive(local_dir: Path, drive_dir: Path, label: str) -> None:
    """Copy all .parquet files under local_dir to drive_dir (Colab only); skip if already present."""
    if not IS_COLAB:
        return
    if drive_dir.exists() and list(drive_dir.rglob("*.parquet")):
        print(f"⏭  {label} already on Drive.")
        return
    drive_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    for f in local_dir.rglob("*.parquet"):
        d = drive_dir / f.relative_to(local_dir).parent
        d.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, d / f.name)
    print(f"✓  {label} synced to Drive in {time.time()-t0:.0f}s")

Mounted at /content/drive
  ✓  origination: /content/drive/MyDrive/master_thesis/data/parquet/origination
  ✓  performance (vintage): /content/drive/MyDrive/master_thesis/data/parquet/performance
  ✓  performance (by period): /content/drive/MyDrive/master_thesis/data/parquet/performance_by_period

  Local precomp : /content/precomp
  Local panel   : /content/panel_final
  Drive panel   : /content/drive/MyDrive/master_thesis/data/panel


### 0.4 · DuckDB Connection & Views


In [ ]:
N_THREADS    = min(40, max(1, (os.cpu_count() or 4) - 4))
MEMORY_LIMIT = "160GB"

def connect_duckdb() -> duckdb.DuckDBPyConnection:
    con = duckdb.connect()
    con.execute(f"SET threads        = {N_THREADS}")
    con.execute(f"SET memory_limit   = '{MEMORY_LIMIT}'")
    con.execute(f"SET temp_directory = '{DUCKDB_TEMP}'")
    con.execute( "SET enable_progress_bar = true")
    return con

con = connect_duckdb()
con.execute(f"CREATE OR REPLACE VIEW sf_origination           AS SELECT * FROM read_parquet('{orig_glob}',        union_by_name=True)")
con.execute(f"CREATE OR REPLACE VIEW sf_performance           AS SELECT * FROM read_parquet('{perf_glob}',        union_by_name=True)")
con.execute(f"CREATE OR REPLACE VIEW sf_performance_by_period AS SELECT * FROM read_parquet('{perf_period_glob}', union_by_name=True)")

orig_n = con.execute("SELECT COUNT(*) FROM sf_origination").fetchone()[0]
perf_n = con.execute("SELECT COUNT(*) FROM sf_performance_by_period").fetchone()[0]
print(f"DuckDB {duckdb.__version__}  |  {N_THREADS} threads  |  {MEMORY_LIMIT}")
print(f"Origination rows : {orig_n:>15,}")
print(f"Performance rows : {perf_n:>15,}")

DuckDB 1.5.4  |  40 threads  |  160GB
Origination rows :      48,597,637
Performance rows :   2,800,558,227


### 0.5 · Release Metadata Verification

Loads the ingestion manifest written by NB01 to verify this pipeline is operating on the intended data snapshot.

In [ ]:
ingestion_manifest_path = MANIFEST_DIR / "ingestion_manifest.json"
if ingestion_manifest_path.exists():
    with open(ingestion_manifest_path) as fh:
        INGESTION_MANIFEST = json.load(fh)
    print("✓  Ingestion manifest loaded:")
    for k in ["release_number","release_date","origination_cutoff_date",
               "performance_cutoff_date","total_origination_rows","total_performance_rows"]:
        print(f"  {k:<45}: {INGESTION_MANIFEST.get(k,'N/A')}")
else:
    print("⚠  ingestion_manifest.json not found - run NB01 first.")
    INGESTION_MANIFEST = {}

✓  Ingestion manifest loaded:
  release_number                               : 46
  release_date                                 : 2026-01-28
  origination_cutoff_date                      : 2025-09-30
  performance_cutoff_date                      : 2025-09-30
  total_origination_rows                       : 48597637
  total_performance_rows                       : 2800558227


### 0.6 · Split Windows, Feature-Eligibility Rules & Target Definition

`SPLIT_WINDOWS` is the authoritative source of temporal boundaries for panel construction. Downstream stages inherit these boundaries through the frozen panel and upstream artefacts or manifests.

**Target variable:** `y = 1` if the loan enters **60+ DPD or REO Acquisition** in any month in `(t, t+12]` - numeric delinquency codes 2+ (60-89 DPD and worse) or REO Acquisition codes `'RA'` (post-Release 29) and legacy `'R'`. Both form the positive class; throughout the thesis this is treated as a delinquency/distress event rather than completed default. See the Purpose section for the panel's event-history interpretation and the learn-predict separation between features observed at or before `t` and the label observed in `(t, t+12]` (Kaufman et al., 2012).

**Effective model scope:** a transition-risk model - it predicts whether loans currently active and at current or 30-DPD status transition into 60+ DPD or REO Acquisition within 12 months. Loans already at 60+ DPD at observation time are excluded from the base population.

**Train vs fitted-train distinction:** the declared `train` split runs through **2004-12-31** so the panel preserves those rows for downstream reproducibility, but NB04 fits the booster on **`obs_year < 2004`** and uses **`obs_year == 2004`** for early-stopping validation. `split='train'` is therefore not identical to the effective fitted-training population.

**Calibration-window note:** `calibration` ends at **2006-12-31**. Observation months from **January 2006 onward** have 12-month forward label windows reaching into 2007. Check 2 quantifies the affected share (51.8%) and the realised positive-rate difference (−0.0183 pp against the clean-half counterfactual); that difference does **not** determine the effect on the conformal threshold q̂, which depends on the joint distribution of scores and labels rather than on the positive rate alone.

**Trimmed `test_normal` window:** `test_normal` ends at **2019-09-30**, not `2019-12-31`. This reduces but does not eliminate COVID-era forward-window contamination: 2019 observation months still have label windows extending into 2020.

**Accounting-cycle realignment (May 2019):** For accounting cycles through April 2019, Freddie Mac's accounting cycle ran from the 16th of a month to the 15th of the following month; for May 2019 onward, the accounting, default-reporting, and forbearance-reporting periods are aligned to the calendar month (Freddie Mac, 2026b). Release 24 also removed a one-month delinquency-assignment offset (Freddie Mac, 2026c). This falls within `test_normal` and does not change the MBA delinquency buckets used to construct the target, but it is a minor reporting-regime boundary.

**COVID test-window note:** `test_covid` is both a macro stress regime and a policy/reporting-regime shift. CARES Act forbearance rules required servicers of federally backed mortgage loans to grant forbearance to eligible borrowers attesting COVID-related hardship, and Freddie Mac loans fall within that universe (CFPB & CSBS, 2020; FHFA Office of Inspector General, 2020). Forbearance and workout policies can alter the observed path from payment difficulty to reported delinquency, even when policy indicators are retained only as metadata.

---

**CP exchangeability caveat:** Standard conformal-prediction coverage guarantees rest on exchangeability of the calibration and test observations (Vovk et al., 2005; Shafer & Vovk, 2008). This panel does not meet that benchmark cleanly for two distinct reasons:

1. *Within-loan temporal correlation.* The same loan appears once per active observation month, so the panel contains repeated observations on the same cross-sectional unit, sharing persistent borrower, loan, and property characteristics (Wooldridge, 2010).
2. *Distributional shift across time.* The calibration split (2005-2006) and the test splits (2007-2023) span materially different credit, macroeconomic, policy, and reporting regimes.

Distributional shift directly breaks the calibration-to-test exchangeability benchmark;
within-loan repetition adds serial dependence. Dependence itself is not synonymous with
non-exchangeability, but both structures matter for the empirical reliability audit
(Foygel Barber et al., 2023).

Time-based splits prevent the same loan-month row from appearing in both calibration and test sets, but do not restore exchangeability across regimes. The thesis contribution is therefore an empirical investigation of how standard CP behaves when its exchangeability condition is knowingly stressed, not a proof of conformal validity under the SFLLD panel structure.

---

**Train-calibration boundary:** the declared train split ends on 2004-12-31 and calibration starts on 2005-01-01 with no temporal gap. A loan active in December 2004 can reappear in January 2005, so the same loan may generate observations on both sides of the boundary; because repeated loan-month observations are dependent panel observations (Wooldridge, 2010), LightGBM scores for adjacent observations of the same loan may be correlated. A burn-in gap would reduce adjacent within-loan dependence but is not imposed.
The resulting cross-boundary dependence is retained as part of the panel structure
studied downstream rather than treated as a separately bounded effect.

In [ ]:
# Time-based split design:
# Splits are chronological rather than random to avoid mixing macro/policy regimes
# and to respect the temporal dependence structure of the panel
# (Roberts et al., 2017). These split boundaries are design choices, not SFLLD fields.
SPLIT_WINDOWS = {
    "train":          ("1999-01-01", "2004-12-31"),
    "calibration":    ("2005-01-01", "2006-12-31"),
    "test_subprime":  ("2007-01-01", "2012-12-31"),
    "test_normal":    ("2013-01-01", "2019-09-30"),   # reduces, but does not eliminate, COVID forward-window contamination
    "test_covid":     ("2020-03-01", "2021-09-30"),
    "test_rate_hike": ("2022-01-01", "2023-12-31"),
}

PROCESS_YEARS     = list(range(1999, 2024))   # 25 years
ALPHA             = 0.10

# Pragmatic distinct-loan viability threshold for §5 Mondrian CP cell-size checks - not a formal
# validity requirement. Keeps binomial noise ≈ sqrt(alpha*(1-alpha)/500) ≈ 1.3 pp.
# Note: NB05b's MONDRIAN_MIN_ROWS = 1000 measures cp_cal rows (a different unit); the two need not match.
MONDRIAN_MIN_CELL = 500

# ── Positive-class condition ─────────────────────────────────────────
POS_CLASS = (
    "TRY_CAST(current_loan_delinquency_status AS INTEGER) >= 2 "
    "OR current_loan_delinquency_status IN ('RA', 'R')"
)

# ── Feature-policy note: interest-rate levels ─────────────────────────────────
# NB02 §5.9 measures the distribution shift of origination features across regimes
# (PSI vs the calibration window) and flags rate-level features dynamically when their
# shift is substantial. original_interest_rate and current_interest_rate are retained
# as active feature candidates DESPITE being regime-bound level variables, because:
#   (a) within-window, rate levels carry cross-sectional risk ordering information, and
#   (b) the thesis intentionally studies CP behaviour under covariate shift rather than
#       engineering the shift away (shift-invariant transforms such as spreads to a
#       contemporaneous market rate would change the research object).
# This is a deliberate design decision; the resulting transport risk is part of the
# distribution-shift phenomenon, not an oversight.

# ── Hard feature exclusions ───────────────────────────────────────────────────
HARD_EXCLUDED_MODEL_FEATURES = [
    "channel",
    "modification_flag",
    "payment_deferral_flag",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "amortization_ratio",
    "deferred_upb_ratio",
    "in_workout_plan_flag",
]

# ── Rebuild flags ─────────────────────────────────────────────────────────────
# Default False - existing artifacts are reused (idempotent run).
# Set to True only when upstream data or feature policy changes require a full rebuild;
# doing so will delete cached precomp/panel/manifest artifacts before regenerating them.
FORCE_REBUILD_PRECOMP_ORIG   = False
FORCE_REBUILD_PRECOMP_LABEL  = False
FORCE_REBUILD_PRECOMP_PREPAY = False
FORCE_REBUILD_PANEL          = False
FORCE_REBUILD_MANIFEST       = False

print("Split windows:")
for split, (start, end) in SPLIT_WINDOWS.items():
    print(f"  {split:<18}: {start}  ->  {end}")

print(f"\nPositive-class condition (60+ DPD or REO Acquisition):")
print(f"  {POS_CLASS}")

print(f"\nHard feature exclusions ({len(HARD_EXCLUDED_MODEL_FEATURES)}):")
for f in HARD_EXCLUDED_MODEL_FEATURES:
    print(f"  - {f}")

print(f"\nProcess years: {PROCESS_YEARS[0]}-{PROCESS_YEARS[-1]}  ({len(PROCESS_YEARS)} years)")

Split windows:
  train             : 1999-01-01  ->  2004-12-31
  calibration       : 2005-01-01  ->  2006-12-31
  test_subprime     : 2007-01-01  ->  2012-12-31
  test_normal       : 2013-01-01  ->  2019-09-30
  test_covid        : 2020-03-01  ->  2021-09-30
  test_rate_hike    : 2022-01-01  ->  2023-12-31

Positive-class condition (60+ DPD or REO Acquisition):
  TRY_CAST(current_loan_delinquency_status AS INTEGER) >= 2 OR current_loan_delinquency_status IN ('RA', 'R')

Hard feature exclusions (8):
  - channel
  - modification_flag
  - payment_deferral_flag
  - loan_age
  - remaining_months_to_legal_maturity
  - amortization_ratio
  - deferred_upb_ratio
  - in_workout_plan_flag

Process years: 1999–2023  (25 years)


---
## Section 1 · Lookup Tables

### 1.1 · Census Division Mapping

Maps U.S. state codes to the nine U.S. Census Bureau divisions, used as one of the later Mondrian CP subgroup dimensions. Census division is preferred over MSA because NB02 §5.7 confirmed substantial MSA missingness in the origination file, while state is much more stable for broad regional grouping.

The 50 states and DC follow the official Census region/division map (U.S. Census Bureau, n.d.). U.S. territories in the SFLLD extract (`PR`, `VI`, `GU`) are **not official Census-division members in the same sense as the 50 states and DC**. They are assigned to nearby broad divisions here purely as a modeling convention to avoid a sparse residual `Unknown` subgroup. This convention must not be described as an official Census classification.

In [ ]:
# 50 states + DC: official Census divisions. PR/VI/GU: pragmatic assignments.
STATE_TO_DIVISION = {
    "CT":"New England",    "ME":"New England",     "MA":"New England",
    "NH":"New England",    "RI":"New England",      "VT":"New England",
    "NJ":"Middle Atlantic","NY":"Middle Atlantic",  "PA":"Middle Atlantic",
    "IN":"E North Central","IL":"E North Central",  "MI":"E North Central",
    "OH":"E North Central","WI":"E North Central",
    "IA":"W North Central","KS":"W North Central",  "MN":"W North Central",
    "MO":"W North Central","NE":"W North Central",  "ND":"W North Central",
    "SD":"W North Central",
    "DE":"South Atlantic", "FL":"South Atlantic",   "GA":"South Atlantic",
    "MD":"South Atlantic", "NC":"South Atlantic",   "SC":"South Atlantic",
    "VA":"South Atlantic", "DC":"South Atlantic",   "WV":"South Atlantic",
    "PR":"South Atlantic", "VI":"South Atlantic",
    "AL":"E South Central","KY":"E South Central",  "MS":"E South Central",
    "TN":"E South Central",
    "AR":"W South Central","LA":"W South Central",  "OK":"W South Central",
    "TX":"W South Central",
    "AZ":"Mountain",       "CO":"Mountain",         "ID":"Mountain",
    "MT":"Mountain",       "NM":"Mountain",         "NV":"Mountain",
    "UT":"Mountain",       "WY":"Mountain",
    "AK":"Pacific",        "CA":"Pacific",           "HI":"Pacific",
    "OR":"Pacific",        "WA":"Pacific",           "GU":"Pacific",
}

division_df = pd.DataFrame(
    [{"property_state": k, "census_division": v} for k, v in STATE_TO_DIVISION.items()]
)
con.execute("CREATE OR REPLACE TEMP TABLE census_division_map AS SELECT * FROM division_df")

unmapped = con.execute("""
    SELECT DISTINCT o.property_state
    FROM sf_origination o
    LEFT JOIN census_division_map m USING (property_state)
    WHERE m.census_division IS NULL AND o.property_state IS NOT NULL
""").df()

if len(unmapped) == 0:
    print(f"✓  Census division map: {len(STATE_TO_DIVISION)} states - all origination states mapped.")
else:
    print(f"⚠  Unmapped states: {unmapped['property_state'].tolist()} - add to STATE_TO_DIVISION.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓  Census division map: 54 states - all origination states mapped.


### 1.2 · Origination-Quarter Vintage Parsing

The true origination quarter is extracted from the `loan_sequence_number`,
not from `first_payment_date`, which can lag origination by several months
(Freddie Mac, 2026a FAQ Q37).

The current documented format is `PYYQnXXXXXXX` (12 characters): product code `P`
(F=FRM, A=ARM), 2-digit year `YY`, literal `Q`, quarter digit `n`, and a 7-digit
random suffix `XXXXXXX` (Freddie Mac, 2026b). Release 27 (April 2021) adjusted the
loan identifier to allow more than 999 999 loans per quarter by expanding the random
suffix from 6 to 7 digits; for loans added in earlier releases, the suffix may be
6 digits (FAQ Q6). The year and quarter positions (characters 2-3 and 5) are
identical in both formats, so the `SUBSTR`-based parsing below is stable across the
full dataset history. The end-of-section assertion confirms no parsing failures.

This section creates the `origination_semantics` temporary view, which adds
`true_vintage_year`, `true_vintage_quarter`, and `origination_rule_group` to every
origination record. `origination_rule_group` classifies each loan as
`'pre_2018q2_non_relief'`, `'relief_refinance'`, or `'2018q2plus_non_relief'`,
which determines the valid disclosure ranges for LTV and CLTV in §3.1.

**No imputation is performed here.** Credit-score/LTV/CLTV/DTI sentinel handling and missingness flags are defined in §3.1.

An assertion validates that no origination records fail vintage parsing (i.e.,
`true_vintage_year` and `true_vintage_quarter` are non-null for all loans).


In [ ]:
print("Parsing true origination quarter...")
t0 = time.time()

con.execute("""
CREATE OR REPLACE TEMP VIEW origination_semantics AS
WITH parsed AS (
    SELECT
        o.*,
        TRY_CAST(SUBSTR(o.loan_sequence_number, 2, 2) AS INTEGER) AS seq_yy,
        TRY_CAST(SUBSTR(o.loan_sequence_number, 5, 1) AS INTEGER) AS seq_q
    FROM sf_origination o
)
SELECT
    *,
    CASE
        WHEN seq_yy BETWEEN 90 AND 99 THEN 1900 + seq_yy
        WHEN seq_yy BETWEEN 0  AND 89 THEN 2000 + seq_yy
        ELSE NULL
    END AS true_vintage_year,
    CASE WHEN seq_q BETWEEN 1 AND 4 THEN seq_q ELSE NULL END AS true_vintage_quarter,
    CASE
        WHEN TRIM(relief_refinance_indicator) = 'Y' THEN 'relief_refinance'
        WHEN (
            CASE WHEN seq_yy BETWEEN 90 AND 99 THEN 1900 + seq_yy
                 WHEN seq_yy BETWEEN 0  AND 89 THEN 2000 + seq_yy
                 ELSE NULL END
        ) < 2018
          OR (
            (CASE WHEN seq_yy BETWEEN 90 AND 99 THEN 1900 + seq_yy
                  WHEN seq_yy BETWEEN 0  AND 89 THEN 2000 + seq_yy
                  ELSE NULL END) = 2018
            AND seq_q = 1
          )
        THEN 'pre_2018q2_non_relief'
        ELSE '2018q2plus_non_relief'
    END AS origination_rule_group
FROM parsed
""")

parse_qc = con.execute("""
SELECT COUNT(*),
       COUNT(*) FILTER (WHERE true_vintage_year IS NULL),
       COUNT(*) FILTER (WHERE true_vintage_quarter IS NULL)
FROM origination_semantics
""").fetchone()
assert parse_qc[1] == 0 and parse_qc[2] == 0, (
    f"Failed to parse true origination vintage: null_year={parse_qc[1]}, null_q={parse_qc[2]}"
)

print(f"✓  origination_semantics view ready: {parse_qc[0]:,} loans")
print("  true_vintage_year and true_vintage_quarter: 0 nulls")

Parsing true origination quarter...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓  origination_semantics view ready: 48,597,637 loans
  true_vintage_year and true_vintage_quarter: 0 nulls


---
## Section 2 · Base Population Diagnostics

Audits the raw size of the base population - active, current or 30-DPD loan-months - split by observation year before the label or feature joins.

**Base population criteria at each observation month `t`:**

1. `zero_balance_code IS NULL` - no termination/disposition event has been recorded for the loan yet. Freddie Mac's Zero Balance Code is set at the monthly reporting period corresponding to the zero balance effective date (the moment the loan balance is reduced to zero). A `NULL` ZBC means no such event has occurred. Note that this condition is **necessary but not sufficient** to identify genuinely active loans: since Release 29, performance records exist for loans in REO status between REO Acquisition and REO Disposition, and these loans also carry a `NULL` ZBC (the balance is not zeroed until disposal, ZBC = `09`) alongside delinquency status `'RA'`. These REO-in-limbo loans are excluded by condition 2 below, not by this condition (Freddie Mac, 2026b, 2026c).

2. `current_loan_delinquency_status IN ('0','00','1')` - the loan is current or 30-59 DPD at observation time. The User Guide defines `0` as current or less than 30 days delinquent, `1` as 30-59 days delinquent, `2` as 60-89 days delinquent, and higher values as later delinquency buckets; REO Acquisition is represented by a status code rather than a numeric DPD bucket (Freddie Mac, 2026b). Loans already at 60+ DPD or REO Acquisition at `t` are excluded because the model predicts onset of serious distress from current/early-delinquency states, not continuation of an already-distressed state. The value `'00'` is retained defensively against raw field-width artifacts in older files after NB01 normalization.

3. `loan_age >= 1` - excludes age-0 records. This is a **modeling decision**, not a data-validity requirement. Under Freddie Mac's loan-age calculation, `loan_age = 0` is one month before the applicable first-payment date; it is not synonymous with the origination month.

   **UPB masking caveat:** Current Actual UPB can be rounded through loan age 6 before modification, so the `loan_age >= 1` filter does not remove all early masked observations. `upb_rel_change_3m` is therefore separately guarded by `loan_age >= 10` and modification status.

In [ ]:
def get_split_label(year: int) -> str:
    """
    Map a reporting year to a split label for the year-level audit table.
    This is a year-granularity approximation used only in this diagnostic.
    """
    for split, (start, end) in SPLIT_WINDOWS.items():
        if int(start[:4]) <= year <= int(end[:4]):
            return split
    return "buffer"

print("Auditing base population by year...")
t0 = time.time()
base_pop_audit = con.execute(f"""
SELECT reporting_year, COUNT(*) AS n_loan_months
FROM sf_performance_by_period
WHERE zero_balance_code IS NULL
  AND loan_age >= 1
  AND current_loan_delinquency_status IN ('0', '00', '1')  -- '00': defensive for historical zero-padded encoding variants in older raw files
  AND reporting_year BETWEEN 1999 AND 2023
GROUP BY 1 ORDER BY 1
""").df()

base_pop_audit["split"] = base_pop_audit["reporting_year"].map(get_split_label)
print(f"Done in {time.time()-t0:.0f}s\n")
print(base_pop_audit.to_string(index=False))
print(f"\nTotal base population loan-months: {base_pop_audit['n_loan_months'].sum():,}")

Auditing base population by year...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Done in 237s

 reporting_year  n_loan_months          split
           1999        6333772          train
           2000       15550109          train
           2001       27564737          train
           2002       49595544          train
           2003       67293639          train
           2004       81753843          train
           2005       86482468    calibration
           2006       92861653    calibration
           2007       98051836  test_subprime
           2008      104261511  test_subprime
           2009      106881627  test_subprime
           2010      110025719  test_subprime
           2011      109504404  test_subprime
           2012      104518905  test_subprime
           2013      104135124    test_normal
           2014      106689556    test_normal
           2015      109147380    test_normal
           2016      111640517    test_normal
           2017      114767136    test_normal
           2018      117238740    test_normal
           2019     

---
## Section 3 · Precomputed Lookup Tables

### Why Precomputation Is Necessary

A naïve panel construction repeatedly joins each observation month `t` to 12-month history and future ranges. At this scale, those repeated temporal range joins were computationally expensive.

**Solution:** materialise three lookup tables **once** before the year loop. The year loop then uses equality joins against those precomputed tables.

| Table | Technique | Purpose |
|---|---|---|
| `precomp_orig` | Single origination scan | Static loan features, cleaned to correct disclosure semantics |
| `precomp_label` | Backward explosion × 12 offsets | `(loan, obs_month)` pairs where `y=1` |
| `precomp_lagged` | Forward explosion × 12 offsets + GROUP BY | Pre-aggregated 12-month lagged DPD features |

**Backward explosion (label):** A positive event at month `m` means `y=1` for observation
months `t ∈ [m-12, m-1]`. For each bad event, emit 12 shifted observation months.

**Forward explosion (lagged features):** A delinquency event at month `m` falls in the
lookback window `[t-12, t-1]` for `t ∈ [m+1, m+12]`. For each delinquency event,
emit 12 shifted observation months.


### 3.1 · Precompute: Origination Features (`precomp_orig`)

Materialises one row per loan with:
- Cleaned origination features with correct disclosure-regime validation
- Explicit sentinel-state indicators for FICO/LTV/CLTV/DTI missingness
- `number_of_units`: `99` (Not Available) set to NULL; valid values 1-4 retained
- True origination-quarter metadata (from loan_sequence_number, not first_payment_date)
- Census division region assignment
- Binary `is_multi_borrower` transform to harmonise the pre/post-2018Q2
  borrower-count cardinality change; see the frozen-artifact note below.
- Corrected HARP flag (vintage 2009-2018 only; post-2018 Relief Refi is not HARP)

**Frozen-artifact note:** Freddie Mac defines `number_of_borrowers = 99` as Not
Available. The executed binary transform uses the numeric rule `> 1`, so these
records map to `is_multi_borrower = 1`. NB02 identifies 6,949 such source records
(0.014% of Release-46 originations). The frozen downstream artifacts are retained; the exact numerical effect of this 6,949-record edge case is not re-estimated.

**DTI:** `original_dti` is NULL when `DTI = 999`; flags `dti_missing`, `dti_missing_relief_refi`, and `dti_missing_non_relief_refi` carry the structure (no imputation).

**Excluded origination fields:** `mortgage_insurance_pct`, `super_conforming_flag`, `seller_name`, `servicer_name`, `postal_code`, `maturity_date`, and `pre_relief_refi_loan_sequence_number` are not selected for the frozen model specification. The downstream model uses the feature set documented in the manifest and thesis Appendix B.4.


In [ ]:
PRECOMP_ORIG_PATH = PRECOMP_DIR / "origination_features.parquet"

if FORCE_REBUILD_PRECOMP_ORIG and PRECOMP_ORIG_PATH.exists():
    PRECOMP_ORIG_PATH.unlink()
    print("⚠  FORCE_REBUILD_PRECOMP_ORIG=True - deleted cached precomp_orig so it can be rebuilt.")

if PRECOMP_ORIG_PATH.exists():
    print("⏭  precomp_orig already exists on local NVMe - skipping.")
else:
    print("Building precomp_orig...")
    t0 = time.time()
    con.execute(f"""
    COPY (
        WITH cleaned AS (
            SELECT
                o.loan_sequence_number,
                o.true_vintage_year    AS vintage_year,
                o.true_vintage_quarter AS vintage_quarter,
                o.origination_rule_group,

                -- Relief Refinance indicator (Y/blank)
                CASE WHEN TRIM(o.relief_refinance_indicator) = 'Y' THEN 1 ELSE 0 END
                    AS relief_refi_flag,

                -- HARP flag: Relief Refi + original LTV > 80 + originated 2009-2018.
                -- This follows Freddie Mac's FAQ definition of the HARP population:
                -- Relief Refinance loans originated through the program from 2009 to 2018
                -- with original LTV above 80 (Freddie Mac, 2026a).
                -- Post-2018 Relief Refinance is not treated as HARP here.
                CASE
                    WHEN TRIM(o.relief_refinance_indicator) = 'Y'
                     AND o.original_loan_to_value != 999
                     AND o.original_loan_to_value > 80
                     AND o.true_vintage_year BETWEEN 2009 AND 2018
                    THEN 1 ELSE 0
                END AS harp_flag,

                -- Credit score: preserve documented observed scores; 9999 / out-of-range values
                -- are treated as NULL rather than imputed (Freddie Mac, 2026a, 2026b).
                -- The January 2026 User Guide documents 300-850, while the FAQ and Release
                -- Notes use 301-850; Release-46 EDA contains observed 300 values.
                -- The executed rule therefore retains 300-850.
                CASE WHEN o.credit_score BETWEEN 300 AND 850
                     THEN CAST(o.credit_score AS DOUBLE) ELSE NULL
                END AS observed_credit_score,

                -- LTV: documented valid range is disclosure-regime dependent.
                -- pre-2018Q2 non-relief: 6-105; relief / 2018Q2+ non-relief: 1-998.
                -- 999 and regime-invalid values are preserved as NULL (Freddie Mac, 2026a, 2026b).
                CASE
                    WHEN o.original_loan_to_value = 999 THEN NULL
                    WHEN o.origination_rule_group = 'pre_2018q2_non_relief'
                         AND o.original_loan_to_value BETWEEN 6 AND 105
                    THEN CAST(o.original_loan_to_value AS DOUBLE)
                    WHEN o.origination_rule_group IN ('relief_refinance','2018q2plus_non_relief')
                         AND o.original_loan_to_value BETWEEN 1 AND 998
                    THEN CAST(o.original_loan_to_value AS DOUBLE)
                    ELSE NULL
                END AS observed_ltv,

                -- CLTV: documented valid range is disclosure-regime dependent.
                -- pre-2018Q2 non-relief upper bound is 200, not 105; relief / 2018Q2+
                -- non-relief range is 1-998. 999 and regime-invalid values are preserved
                -- as NULL (Freddie Mac, 2026a, 2026b).
                CASE
                    WHEN o.original_combined_loan_to_value = 999 THEN NULL
                    WHEN o.origination_rule_group = 'pre_2018q2_non_relief'
                         AND o.original_combined_loan_to_value BETWEEN 6 AND 200
                    THEN CAST(o.original_combined_loan_to_value AS DOUBLE)
                    WHEN o.origination_rule_group IN ('relief_refinance','2018q2plus_non_relief')
                         AND o.original_combined_loan_to_value BETWEEN 1 AND 998
                    THEN CAST(o.original_combined_loan_to_value AS DOUBLE)
                    ELSE NULL
                END AS observed_cltv,

                -- DTI: only valid observed values (0, 65] are retained.
                -- DTI=999 conflates >65% censoring, genuinely unavailable values, and Relief
                -- Refinance masking. Imputing a median would fabricate numeric underwriting
                -- information; explicit missingness flags carry the structural signal instead
                -- (Freddie Mac, 2026a, 2026b; Rubin, 1976; Little & Rubin, 2019).
                CASE
                    WHEN o.original_debt_to_income_ratio > 0
                     AND o.original_debt_to_income_ratio <= 65
                    THEN CAST(o.original_debt_to_income_ratio AS DOUBLE)
                    ELSE NULL
                END AS observed_dti,

                CASE WHEN TRY_CAST(o.original_interest_rate AS DOUBLE) > 0
                     THEN CAST(o.original_interest_rate AS DOUBLE) ELSE NULL
                END AS original_interest_rate_clean,

                CASE WHEN TRY_CAST(o.original_upb AS DOUBLE) > 0
                     THEN CAST(o.original_upb AS DOUBLE) ELSE NULL
                END AS original_upb_clean,

                CASE WHEN TRY_CAST(o.original_loan_term AS INTEGER) > 0
                     THEN CAST(o.original_loan_term AS INTEGER) ELSE NULL
                END AS original_loan_term_clean,

                -- Categorical features with explicit unknown handling
                CASE WHEN TRIM(o.loan_purpose)       IN ('9','')      THEN 'Unknown'
                     ELSE COALESCE(TRIM(o.loan_purpose), 'Unknown')   END AS loan_purpose,

                -- channel: B and C codes are reliable only from 2008+; pre-2008 originations
                -- are dominated by 'T' (TPO Not Specified) because Freddie Mac did not collect
                -- granular broker/correspondent/retail channel information before 2008
                -- (Freddie Mac, 2026a). Pre-2008 values do not constitute a stable economic
                -- channel signal. This encoding artifact creates a cross-vintage inconsistency.
                CASE WHEN TRIM(o.channel)             IN ('9','')      THEN 'Unknown'
                     ELSE COALESCE(TRIM(o.channel), 'Unknown')         END AS channel,

                CASE WHEN TRIM(o.occupancy_status)    = '9'            THEN 'Unknown'
                     ELSE COALESCE(TRIM(o.occupancy_status), 'Unknown') END AS occupancy_status,

                CASE WHEN TRIM(o.property_type)       = '99'           THEN 'Unknown'
                     ELSE COALESCE(TRIM(o.property_type), 'Unknown')    END AS property_type,

                -- number_of_units: valid values 1-4; 99 = Not Available (sentinel, not ordinal).
                -- Treat 99 as NULL consistent with the FICO/LTV/CLTV/DTI sentinel approach.
                CASE WHEN TRY_CAST(o.number_of_units AS INTEGER) IN (1, 2, 3, 4)
                     THEN TRY_CAST(o.number_of_units AS INTEGER)
                     ELSE NULL
                END AS number_of_units,

                -- is_multi_borrower: binary transform intended to harmonise borrower-count semantics.
                -- Raw number_of_borrowers changes cardinality in 2018Q2:
                --   pre-2018Q2: 01=single, 02=MORE THAN ONE
                --   2018Q2+: exact counts 01-10
                -- Frozen-artifact note: 99=Not Available is numerically >1 and is therefore
                -- encoded as 1 by the executed rule; see §3.1 markdown for prevalence/context.
                CASE WHEN TRY_CAST(o.number_of_borrowers AS INTEGER) > 1 THEN 1 ELSE 0 END
                    AS is_multi_borrower,

                -- first_time_homebuyer_flag: frozen analytical encoding.
                --   NULL/blank -> Not_Applicable
                --   '9'        -> Unknown
                -- FAQ Q38 supports blank as not applicable for refinance, investment-property,
                -- and second-home loans. The data dictionary combines code 9 as
                -- "Not Available or Not Applicable", so the blank/9 distinction is an
                -- analytical encoding rather than a fully source-identified distinction.
                CASE
                    WHEN TRIM(o.first_time_homebuyer_flag) = 'Y'   THEN 'Y'
                    WHEN TRIM(o.first_time_homebuyer_flag) = 'N'   THEN 'N'
                    WHEN TRIM(o.first_time_homebuyer_flag) = '9'   THEN 'Unknown'
                    WHEN o.first_time_homebuyer_flag IS NULL
                      OR TRIM(o.first_time_homebuyer_flag) = ''    THEN 'Not_Applicable'
                    ELSE 'Unknown'
                END AS first_time_homebuyer_flag,

                o.property_state

            FROM origination_semantics o
            WHERE o.true_vintage_year IS NOT NULL
              AND o.true_vintage_quarter IS NOT NULL
        )

        SELECT
            c.loan_sequence_number,
            c.vintage_year,
            c.vintage_quarter,
            c.origination_rule_group,

            -- FICO: preserve NULL for disclosure-sentinel / non-observed cases.
            c.observed_credit_score AS credit_score,
            CASE WHEN c.observed_credit_score IS NULL THEN 1 ELSE 0 END AS fico_missing,

            -- LTV: preserve NULL for sentinel / regime-invalid cases.
            c.observed_ltv AS original_ltv,
            CASE WHEN c.observed_ltv IS NULL THEN 1 ELSE 0 END AS ltv_missing,

            -- CLTV: preserve NULL for sentinel / regime-invalid cases.
            c.observed_cltv AS original_cltv,
            CASE WHEN c.observed_cltv IS NULL THEN 1 ELSE 0 END AS cltv_missing,

            -- DTI: preserve NULL when 999 / masked / structurally unobserved.
            c.observed_dti AS original_dti,
            CASE WHEN c.observed_dti IS NULL THEN 1 ELSE 0 END AS dti_missing,
            CASE WHEN c.observed_dti IS NULL AND c.relief_refi_flag = 1 THEN 1 ELSE 0 END
                AS dti_missing_relief_refi,
            CASE WHEN c.observed_dti IS NULL AND c.relief_refi_flag = 0 THEN 1 ELSE 0 END
                AS dti_missing_non_relief_refi,

            c.original_interest_rate_clean AS original_interest_rate,
            c.original_upb_clean           AS original_upb,
            c.original_loan_term_clean     AS original_loan_term,

            c.loan_purpose,
            c.channel,
            c.occupancy_status,
            c.property_type,
            c.number_of_units,
            c.is_multi_borrower,
            c.first_time_homebuyer_flag,
            c.property_state,
            COALESCE(cd.census_division, 'Unknown') AS census_division,
            c.relief_refi_flag,
            c.harp_flag

        FROM cleaned c
        LEFT JOIN census_division_map cd
               ON c.property_state  = cd.property_state
    )
    TO '{PRECOMP_ORIG_PATH.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 500000)
    """)

    elapsed = time.time() - t0
    n_orig = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{PRECOMP_ORIG_PATH.as_posix()}')"
    ).fetchone()[0]
    print(f"✓  precomp_orig: {n_orig:,} loans in {elapsed:.0f}s")

con.execute(f"CREATE OR REPLACE VIEW precomp_orig AS SELECT * FROM read_parquet('{PRECOMP_ORIG_PATH.as_posix()}')")
print("✓  precomp_orig view registered.")

Building precomp_orig...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓  precomp_orig: 48,597,637 loans in 5s
✓  precomp_orig view registered.


#### Drive Sync - `precomp_orig`

In [ ]:
dest = DRIVE_PRECOMP / "origination_features.parquet"
if not dest.exists():
    print("Copying precomp_orig to Drive...")
    shutil.copy2(PRECOMP_ORIG_PATH, dest)
    print(f"✓  Copied to {dest}")
else:
    print("⏭  precomp_orig already on Drive.")

Copying precomp_orig to Drive...
✓  Copied to /content/drive/MyDrive/master_thesis/data/precomp/origination_features.parquet


### 3.2 · Precompute: Forward Label Lookup (`precomp_label`)

Produces all `(loan_sequence_number, obs_month)` pairs where `y = 1`.

The positive event definition is:

- numeric `current_loan_delinquency_status >= 2`, corresponding to 60-89 DPD or worse; or
- `current_loan_delinquency_status IN ('RA', 'R')`, corresponding to REO Acquisition status, with `RA` used after the Release 29 coding update and `R` retained for legacy robustness (Freddie Mac, 2026b, 2026c).

The positive-event lookup deliberately uses `zero_balance_code IS NULL`. This keeps the event definition tied to active monthly-performance states and avoids turning terminal zero-balance records themselves into active observation states. Check 9 later quantifies the remaining fast-default gap for credit-event terminations that may not have displayed an active 60+ DPD / REO status before termination.

In [ ]:
PRECOMP_LABEL_DIR = PRECOMP_DIR / "label"
PRECOMP_LABEL_DIR.mkdir(parents=True, exist_ok=True)

label_glob_path = (PRECOMP_LABEL_DIR / "**" / "*.parquet").as_posix()
label_files = list(PRECOMP_LABEL_DIR.rglob("*.parquet"))

if FORCE_REBUILD_PRECOMP_LABEL and PRECOMP_LABEL_DIR.exists():
    shutil.rmtree(PRECOMP_LABEL_DIR)
    PRECOMP_LABEL_DIR.mkdir(parents=True, exist_ok=True)
    label_files = []
    print("⚠  FORCE_REBUILD_PRECOMP_LABEL=True - deleted cached precomp_label so it can be rebuilt.")

if label_files:
    n_label = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{label_glob_path}', union_by_name=True)"
    ).fetchone()[0]
    print(f"⏭  precomp_label already exists: {n_label:,} rows across {len(label_files)} files - skipping.")
else:
    print("Building precomp_label (backward explosion of positive events)...")
    t0 = time.time()
    con.execute(f"""
    COPY (
    WITH
    positive_events AS (
        -- Positive-class events: 60+ DPD or REO Acquisition.
        -- Includes numeric codes >= 2 AND REO-acquisition servicing-state codes ('RA','R').
        -- Restrict to ACTIVE rows only so labels are sourced from the same active-state
        -- universe used for panel feature construction.
        SELECT
            loan_sequence_number,
            monthly_reporting_period AS event_month
        FROM sf_performance_by_period
        WHERE zero_balance_code IS NULL
          AND (
                TRY_CAST(current_loan_delinquency_status AS INTEGER) >= 2
                OR current_loan_delinquency_status IN ('RA', 'R')
          )
    ),
    exploded AS (
        -- Backward explosion: positive event at m -> obs_months m-1, m-2, ..., m-12
        -- n=1:  obs_month = event_month - 1   (event at t+1)
        -- n=12: obs_month = event_month - 12  (event at t+12, the horizon boundary)
        SELECT
            b.loan_sequence_number,
            b.event_month - (n * INTERVAL '1' MONTH) AS obs_month,
            YEAR(b.event_month - (n * INTERVAL '1' MONTH)) AS obs_year
        FROM positive_events b
        CROSS JOIN generate_series(1, 12) AS gs(n)
        WHERE b.event_month - (n * INTERVAL '1' MONTH) >= DATE '1999-01-01'
          AND b.event_month - (n * INTERVAL '1' MONTH) <= DATE '2023-12-31'
    )
    SELECT DISTINCT loan_sequence_number, obs_month, obs_year, 1 AS y
    FROM exploded
    )
    TO '{PRECOMP_LABEL_DIR.as_posix()}'
    (FORMAT PARQUET, PARTITION_BY (obs_year), COMPRESSION ZSTD,
     ROW_GROUP_SIZE 500000, OVERWRITE_OR_IGNORE TRUE)
    """)

    elapsed = time.time() - t0
    label_files = list(PRECOMP_LABEL_DIR.rglob("*.parquet"))
    n_label = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{label_glob_path}', union_by_name=True)"
    ).fetchone()[0]
    print(f"✓  precomp_label: {n_label:,} rows, {len(label_files)} files")

label_glob = label_glob_path
con.execute(f"CREATE OR REPLACE VIEW precomp_label AS SELECT * FROM read_parquet('{label_glob}', union_by_name=True)")

Building precomp_label (backward explosion of positive events)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓  precomp_label: 77,140,170 rows, 25 files


### 3.2b · Precompute: Prepayment Forward Window Lookup (`precomp_prepay`)

Produces all `(loan_sequence_number, obs_month)` pairs where code-01 payoff/maturity occurs within the 12-month forward label window - i.e., `zero_balance_code = '01'` (Prepaid or Matured) occurs at some month `m ∈ (obs_month, obs_month + 12]`. In the SFLLD this code covers both scheduled-maturity payoffs and early voluntary prepayment.

This table is a **robustness diagnostic only**: it is not joined into the modeling
panel and does not affect training or the primary CP evaluation. Code 01 does not
itself determine `y`: an earlier 60+ DPD/REO event still yields `y = 1`; otherwise
the observation remains `y = 0`. Econometrically, code-01 termination is a competing
risk (Deng et al., 2000). NB06 §8b uses this lookup for a broader sensitivity that
excludes every row with code 01 inside the forward window.

The base-population filter in §3.6 (`zero_balance_code IS NULL`) already excludes loan-months at which prepayment has occurred; the pairs flagged here are loan-months still active at `obs_month` that prepay before `obs_month + 12`.



In [ ]:
PRECOMP_PREPAY_DIR = PRECOMP_DIR / "prepay"
PRECOMP_PREPAY_DIR.mkdir(parents=True, exist_ok=True)

if FORCE_REBUILD_PRECOMP_PREPAY and PRECOMP_PREPAY_DIR.exists():
    shutil.rmtree(PRECOMP_PREPAY_DIR)
    PRECOMP_PREPAY_DIR.mkdir(parents=True, exist_ok=True)
    print("⚠  FORCE_REBUILD_PRECOMP_PREPAY=True - deleted cached precomp_prepay so it can be rebuilt.")

prepay_glob_path = (PRECOMP_PREPAY_DIR / "**" / "*.parquet").as_posix()
prepay_files = list(PRECOMP_PREPAY_DIR.rglob("*.parquet"))

if prepay_files:
    n_prepay = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{prepay_glob_path}', union_by_name=True)"
    ).fetchone()[0]
    print(f"⏭  precomp_prepay already exists: {n_prepay:,} rows across "
          f"{len(prepay_files)} files - skipping.")
else:
    print("Building precomp_prepay (backward explosion of prepayment events)...")
    t0 = time.time()
    con.execute(f"""
    COPY (
    WITH
    prepay_events AS (
        -- Code-01 payoff / maturity events.
        -- This lookup does not determine y: an earlier active 60+ DPD / REO event
        -- can still make the observation y=1. NB06 excludes all rows with code 01
        -- inside the forward horizon as a competing-risk sensitivity.
        SELECT
            loan_sequence_number,
            monthly_reporting_period AS event_month
        FROM sf_performance_by_period
        WHERE zero_balance_code = '01'
    ),
    exploded AS (
        -- Backward explosion: prepayment at m → obs_months m-1, m-2, ..., m-12
        -- These are loan-months where the loan is still active at obs_month but
        -- will prepay before obs_month + 12 months (the label horizon).
        SELECT
            b.loan_sequence_number,
            b.event_month - (n * INTERVAL '1' MONTH)         AS obs_month,
            YEAR(b.event_month - (n * INTERVAL '1' MONTH))   AS obs_year,
            1                                                  AS prepay_within_horizon
        FROM prepay_events b
        CROSS JOIN generate_series(1, 12) gs(n)
        WHERE YEAR(b.event_month - (n * INTERVAL '1' MONTH)) BETWEEN 1999 AND 2023
    )
    SELECT
        loan_sequence_number,
        obs_month,
        obs_year,
        prepay_within_horizon
    FROM exploded
    )
    TO '{PRECOMP_PREPAY_DIR.as_posix()}'
    (FORMAT PARQUET, PARTITION_BY (obs_year), COMPRESSION ZSTD,
     ROW_GROUP_SIZE 250000)
    """)
    elapsed = time.time() - t0

    n_built = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{prepay_glob_path}', union_by_name=True)"
    ).fetchone()[0]
    print(f"✓  precomp_prepay: {n_built:,} rows in {elapsed:.0f}s")

con.execute(
    f"CREATE OR REPLACE VIEW precomp_prepay AS "
    f"SELECT * FROM read_parquet('{prepay_glob_path}', union_by_name=True)"
)

# Spot-check: row counts by obs_year within the split windows
spot = con.execute("""
    SELECT obs_year, COUNT(*) AS n_prepay_pairs
    FROM precomp_prepay
    WHERE obs_year BETWEEN 2005 AND 2023
    GROUP BY 1 ORDER BY 1
""").df()
print("\nPrepayment-within-horizon (loan, obs_month) pairs by obs_year:")
print(spot.to_string(index=False))

# Diagnostic prepayment-lookup counts by split-year range;
# exact retained-row exclusion rates are computed in NB06 §8b.
for split_name, (start, end) in list(SPLIT_WINDOWS.items())[1:]:  # skip train
    yr_lo, yr_hi = int(start[:4]), int(end[:4])
    n_pp = con.execute(
        f"SELECT COUNT(*) FROM precomp_prepay "
        f"WHERE obs_year BETWEEN {yr_lo} AND {yr_hi}"
    ).fetchone()[0]
    print(f"  {split_name:<18}: {n_pp:>12,} prepayment-contaminated (loan, obs_month) pairs")

Building precomp_prepay (backward explosion of prepayment events)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓  precomp_prepay: 397,913,679 rows in 20s

Prepayment-within-horizon (loan, obs_month) pairs by obs_year:
 obs_year  n_prepay_pairs
     2005        10990453
     2006         9172847
     2007         9586211
     2008        15325851
     2009        17590551
     2010        20135838
     2011        22917195
     2012        25665451
     2013        13837551
     2014        13638439
     2015        14942515
     2016        15825484
     2017        13692760
     2018        13133473
     2019        24820326
     2020        38417121
     2021        24097972
     2022        10398394
     2023         9299492
  calibration       :   20,163,300 prepayment-contaminated (loan, obs_month) pairs
  test_subprime     :  111,221,097 prepayment-contaminated (loan, obs_month) pairs
  test_normal       :  109,890,548 prepayment-contaminated (loan, obs_month) pairs
  test_covid        :   62,515,093 prepayment-contaminated (loan, obs_month) pairs
  test_rate_hike    :   19,697,886 prepay

#### Drive Sync - `precomp_prepay`

In [ ]:
sync_dir_to_drive(PRECOMP_PREPAY_DIR, DRIVE_PRECOMP / "prepay", "precomp_prepay")

✓  precomp_prepay synced to Drive in 1s


#### Drive Sync - `precomp_label`

In [ ]:
sync_dir_to_drive(PRECOMP_LABEL_DIR, DRIVE_PRECOMP / "label", "precomp_label")

✓  precomp_label synced to Drive in 3s


### 3.3 · Precompute: Lagged Feature Lookup (`precomp_lagged`)

Produces pre-aggregated 12-month lagged delinquency features for every
`(loan_sequence_number, obs_month)` pair that had any delinquency in its lookback window.
Pairs with clean history do not appear - the year loop assigns clean defaults via COALESCE.

The use of loan-month histories and lagged states reflects the panel structure of the data: repeated observations on the same loan are temporally dependent and should not be treated as independent cross-sectional observations (Wooldridge, 2010).

Lagged delinquency features are retained because recent arrears status is economically and predictively central in mortgage-default modeling. Fitzpatrick and Mues (2016) provide mortgage-default prediction evidence using modern classification methods on loan-level mortgage portfolios, including borrower repayment behaviour, prior arrears, loan modifications, current LTV (as a proxy for equity position), and regional economic conditions as relevant predictive information. The citation is therefore used as mortgage-default prediction context, not as the formal methodological basis for the lagged-panel construction itself.

**Features produced:**
| Feature | Aggregation | Meaning |
|---|---|---|
| `n_times_30dpd_last_12m` | `SUM(is_30dpd)` | 30-DPD months in lookback window |
| `max_dlq_last_12m` | `MAX(dlq_code_num)` | Maximum encoded status; RA/R coded as 99 |
| `months_since_last_dlq` | `MIN(months_elapsed)` | Months since most recent delinquency; **sentinel 999 = no delinquency in window** |

Here, 99 is an encoding sentinel rather than a universal severity maximum; the
Release-46 raw inventory contains numeric delinquency codes above 99.

`months_since_last_dlq = 999` is a deliberate sentinel, not an ordinal lag value.
LightGBM splits on this cleanly (split at < 999 vs = 999 creates an "ever delinquent
in last 12m" split). The manifest documents this sentinel convention for NB04.

**Leakage prevention:** Only `zero_balance_code IS NULL` rows enter the input.
Terminated loan events are not attributed to lookback windows of later observations.
The label window `(t, t+12]` and lagged window `[t-12, t-1]` are strictly non-overlapping.

**History scope:** The model uses only the three 12-month history features above. Serious delinquency more than 12 months before `t` is not represented explicitly, and repeated 60+ DPD months are not counted separately; `max_dlq_last_12m` records only maximum severity. The incremental value of broader-history or dedicated 60+ frequency features was not evaluated.

In [ ]:
PRECOMP_LAGGED_DIR = PRECOMP_DIR / "lagged"
PRECOMP_LAGGED_DIR.mkdir(parents=True, exist_ok=True)

lagged_glob_path = (PRECOMP_LAGGED_DIR / "**" / "*.parquet").as_posix()
lagged_files = list(PRECOMP_LAGGED_DIR.rglob("*.parquet"))

if lagged_files:
    n_lagged = con.execute(f"SELECT COUNT(*) FROM read_parquet('{lagged_glob_path}', union_by_name=True)").fetchone()[0]
    print(f"⏭  precomp_lagged already exists: {n_lagged:,} rows - skipping.")
else:
    print("Building precomp_lagged (forward explosion of delinquency events)...")
    t0 = time.time()
    con.execute(f"""
    COPY (
    WITH
    dlq_events AS (
        -- All delinquency events (code >= 1) on active loans.
        -- RA/R are assigned the sentinel 99 before max aggregation.
        -- Numeric delinquency codes can exceed 99, so 99 is not a universal severity maximum.
        SELECT
            loan_sequence_number,
            monthly_reporting_period AS event_month,
            CASE WHEN current_loan_delinquency_status = '1' THEN 1 ELSE 0 END AS is_30dpd,
            CASE WHEN current_loan_delinquency_status IN ('RA','R') THEN 99
                 ELSE COALESCE(TRY_CAST(current_loan_delinquency_status AS INTEGER), 0)
            END AS dlq_code_num
        FROM sf_performance_by_period
        WHERE (
            TRY_CAST(current_loan_delinquency_status AS INTEGER) >= 1
            OR current_loan_delinquency_status IN ('RA','R')
        )
          AND zero_balance_code IS NULL
    ),
    exploded AS (
        -- Forward explosion: event at m -> obs_months m+1, m+2, ..., m+12
        -- months_elapsed = n = how many months ago this event was relative to obs_month
        SELECT
            d.loan_sequence_number,
            d.event_month + (n * INTERVAL '1' MONTH)       AS obs_month,
            YEAR(d.event_month + (n * INTERVAL '1' MONTH)) AS obs_year,
            d.is_30dpd,
            d.dlq_code_num,
            CAST(n AS INTEGER)                              AS months_elapsed
        FROM dlq_events d
        CROSS JOIN generate_series(1, 12) AS gs(n)
        WHERE d.event_month + (n * INTERVAL '1' MONTH) >= DATE '1999-01-01'
          AND d.event_month + (n * INTERVAL '1' MONTH) <= DATE '2023-12-31'
    )
    SELECT
        loan_sequence_number,
        obs_month,
        obs_year,
        SUM(is_30dpd)     AS n_times_30dpd_last_12m,
        MAX(dlq_code_num) AS max_dlq_last_12m,
        -- MIN gives the nearest (most recent) delinquency event
        MIN(months_elapsed) AS months_since_last_dlq
    FROM exploded
    GROUP BY 1, 2, 3
    )
    TO '{PRECOMP_LAGGED_DIR.as_posix()}'
    (FORMAT PARQUET, PARTITION_BY (obs_year), COMPRESSION ZSTD,
     ROW_GROUP_SIZE 500000, OVERWRITE_OR_IGNORE TRUE)
    """)

    elapsed = time.time() - t0
    lagged_files = list(PRECOMP_LAGGED_DIR.rglob("*.parquet"))
    n_lagged = con.execute(f"SELECT COUNT(*) FROM read_parquet('{lagged_glob_path}', union_by_name=True)").fetchone()[0]
    print(f"✓  precomp_lagged: {n_lagged:,} rows, {len(lagged_files)} files")

lagged_glob = lagged_glob_path
con.execute(f"CREATE OR REPLACE VIEW precomp_lagged AS SELECT * FROM read_parquet('{lagged_glob}', union_by_name=True)")

Building precomp_lagged (forward explosion of delinquency events)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓  precomp_lagged: 165,241,751 rows, 25 files


#### Drive Sync - `precomp_lagged`

In [ ]:
sync_dir_to_drive(PRECOMP_LAGGED_DIR, DRIVE_PRECOMP / "lagged", "precomp_lagged")

✓  precomp_lagged synced to Drive in 1s


### 3.4 · Precomputation Summary

In [ ]:
n_orig   = con.execute("SELECT COUNT(*) FROM precomp_orig").fetchone()[0]
n_label  = con.execute("SELECT COUNT(*) FROM precomp_label").fetchone()[0]
n_lagged = con.execute("SELECT COUNT(*) FROM precomp_lagged").fetchone()[0]
n_prepay = con.execute("SELECT COUNT(*) FROM precomp_prepay").fetchone()[0]

print("=== Precomputed lookup table summary ===")
print(f"  precomp_orig   : {n_orig:>12,} loans (one row per loan)")
print(f"  precomp_label  : {n_label:>12,} (loan, obs_month) pairs where y=1")
print(f"  precomp_lagged : {n_lagged:>12,} (loan, obs_month) pairs with lagged features")
print(f"  precomp_prepay : {n_prepay:>12,} (loan, obs_month) pairs with prepayment-within-horizon flag")

label_by_year = con.execute("""
    SELECT obs_year, COUNT(*) AS n_y1_pairs
    FROM precomp_label WHERE obs_year BETWEEN 1999 AND 2023
    GROUP BY 1 ORDER BY 1
""").df()
print("\ny=1 pairs by obs_year (spot-check):")
print(label_by_year.to_string(index=False))

=== Precomputed lookup table summary ===
  precomp_orig   :   48,597,637 loans (one row per loan)
  precomp_label  :   77,140,170 (loan, obs_month) pairs where y=1
  precomp_lagged :  165,241,751 (loan, obs_month) pairs with lagged features
  precomp_prepay :  397,913,679 (loan, obs_month) pairs with prepayment-within-horizon flag

y=1 pairs by obs_year (spot-check):
 obs_year  n_y1_pairs
     1999       89745
     2000      298458
     2001      637070
     2002     1016194
     2003     1161147
     2004     1252995
     2005     1453488
     2006     1525508
     2007     2203762
     2008     4189002
     2009     6356970
     2010     6493918
     2011     5927434
     2012     5285363
     2013     4361428
     2014     3562249
     2015     2916831
     2016     2600897
     2017     2678772
     2018     2308189
     2019     5249080
     2020     6486706
     2021     3714677
     2022     2683350
     2023     2686937


### 3.5 · Right-Truncation Check

In [ ]:
# Verify that all 12-month forward label windows close within the available performance data.
# If a split's forward window extends beyond the data, observations at the end of that split
# would have their labels right-censored - they appear as y=0 when in fact the outcome is
# unknown. This check must pass before the panel can be considered complete.
max_perf_period = con.execute(
    "SELECT MAX(monthly_reporting_period) FROM sf_performance_by_period"
).fetchone()[0]
print(f"Latest performance period in data: {max_perf_period}")

all_ok = True
for split_name, (start, end) in SPLIT_WINDOWS.items():
    window_close = pd.Timestamp(end) + pd.DateOffset(months=12)
    ok = window_close <= pd.Timestamp(max_perf_period)
    flag = "✓" if ok else "✗ INSUFFICIENT DATA"
    print(f"  {flag}  {split_name:<18}: last obs {end}, label window closes {window_close.date()}")
    if not ok:
        all_ok = False

if not all_ok:
    raise RuntimeError("Forward label windows extend beyond available data. Revise SPLIT_WINDOWS.")
else:
    print("\n✓  All forward label windows are fully covered by the available performance data.")

Latest performance period in data: 2025-09-01
  ✓  train             : last obs 2004-12-31, label window closes 2005-12-31
  ✓  calibration       : last obs 2006-12-31, label window closes 2007-12-31
  ✓  test_subprime     : last obs 2012-12-31, label window closes 2013-12-31
  ✓  test_normal       : last obs 2019-09-30, label window closes 2020-09-30
  ✓  test_covid        : last obs 2021-09-30, label window closes 2022-09-30
  ✓  test_rate_hike    : last obs 2023-12-31, label window closes 2024-12-31

✓  All forward label windows are fully covered by the available performance data.


### 3.6 · Year-by-Year Panel Assembly

With the three precomputed tables registered as views, each observation-year panel is assembled through equality joins only - no inequality range conditions.

**Assembly logic:**

1. **Base population filter:** active, current-or-30-DPD, `loan_age >= 1`
2. **Label join:** LEFT JOIN to `precomp_label` on `(loan, obs_month)` → COALESCE to `y=0`
3. **Lagged features join:** LEFT JOIN to `precomp_lagged` → COALESCE to clean defaults
4. **3-month UPB delta:** reads the prior-year partition for `t-3` balance data
5. **Origination features join:** INNER JOIN to `precomp_orig`
6. **Balance-derived features:** computed inline with masking-period and modification guards

The core panel construction is engineering. The methodological choices embedded in it are: the grouped-/discrete-time transition-risk structure of the loan-month panel (Wooldridge, 2010); the transition-risk base population; the competing-risk treatment of prepayment for the binary target (Deng et al., 2000); the learn-predict separation between features observable at `t` and labels observed in `(t, t+12]` (Kaufman et al., 2012); NULL preservation for disclosure sentinels (Rubin, 1976; Little & Rubin, 2019); exclusion of later-regime features from the frozen historical-training specification; and time-based split construction for temporally dependent panel data (Roberts et al., 2017).

**Important modeling distinction:** not every column assembled into the panel is eligible to become a model feature. The panel intentionally contains three classes of columns:

- **active model candidates** - eligible for NB04 if variable in the training window and not hard-excluded
- **diagnostic / metadata columns** - retained for interpretation, subgrouping, and later-period diagnostics
- **structurally excluded columns** - retained in the panel but never promoted to the active feature list

Hard exclusions are defined in §0.6 (`HARD_EXCLUDED_MODEL_FEATURES`) and reported in Check 6.

**`rate_spread` as a partial modification proxy:** `rate_spread = current_interest_rate - original_interest_rate`. In the SFLLD Standard Dataset (fixed-rate loans only), it departs from zero almost exclusively after a rate modification (an ARM reset is the only other source, and ARMs are out of scope). It is retained as an active feature because it captures the economic *effect* of a rate change, while `modification_flag` itself is hard-excluded; the model is thus allowed to learn the rate-divergence signal without the institutional indicator. It must be disclosed in the thesis methodology as a partial modification proxy - capturing the economic effect of rate modifications, not a clean market-rate variable, and not the Y/P temporal structure of `modification_flag` or the payment-deferral / forbearance channels.

**UPB-dynamics guard:** `upb_rel_change_3m` - the signed 3-month relative UPB change `(UPB_t - UPB_{t-3}) / UPB_{t-3}` - is constructed only for non-modified observations with `loan_age >= 10`, so post-modification balance jumps cannot reintroduce excluded modification-era information. The `loan_age >= 10` threshold ensures the `t-3` look-back also clears the 6-month origination UPB masking window: at observation month with loan age `L`, the `t-3` record has loan age `L-3`, and requiring `L-3 > 6` gives `L ≥ 10`. Per Release 39, `loan_age` resets on modification but not on payment deferral, so a deferred loan with `loan_age <= 6` is genuinely within the origination masking window and the guard suppresses `upb_rel_change_3m` correctly.

**Schema contract:** `vintage_year` and `vintage_quarter` come from `precomp_orig` (parsed loan-sequence semantics) and remain integer-coded in the final panel. A schema assertion below checks this because NB04 expects integer categorical encodings rather than partition strings like `'Q1'`.

**INNER JOIN on `precomp_orig`:** NB02 verifies that no performance loan lacks an origination match, so this join is lossless in the ingested release. Check 7 reconciles the remaining base-to-panel loan reduction to the excluded buffer periods.

The downstream prediction pipeline uses LightGBM-style gradient-boosted decision trees, a scalable GBDT implementation introduced by Ke et al. (2017), who propose gradient-based one-side sampling and exclusive feature bundling to improve training efficiency on large, high-dimensional data.

In [ ]:
def build_panel_year(
    con:          duckdb.DuckDBPyConnection,
    obs_year:     int,
    out_dir:      Path,
    skip_existing: bool = True,
) -> dict:
    """
    Assemble the panel for a single observation year using the precomputed equality joins.
    Returns a summary dict with row counts and positive rate.
    """
    out_file = out_dir / f"obs_year={obs_year}" / "panel.parquet"
    out_file.parent.mkdir(parents=True, exist_ok=True)

    if skip_existing and out_file.exists():
        r = con.execute(
            f"SELECT COUNT(*) AS n, CAST(SUM(y) AS BIGINT) AS pos "
            f"FROM read_parquet('{out_file.as_posix()}')"
        ).fetchone()
        return {
            "year": obs_year, "n_rows": r[0], "n_positive": r[1],
            "positive_rate_pct": round(r[1] / r[0] * 100, 4) if r[0] > 0 else 0.0,
            "status": "skipped",
        }

    y_prev = obs_year - 1
    t0 = time.time()

    con.execute(f"""
    COPY (
        WITH
        -- Base population: active, current or 30-DPD, loan_age >= 1
        -- Age 0 is excluded as a modeling decision, not as invalid SFLLD data.
        base AS (
            SELECT
                p.loan_sequence_number,
                p.monthly_reporting_period                              AS obs_month,
                CAST(p.loan_age AS INTEGER)                             AS loan_age,
                CAST(p.remaining_months_to_legal_maturity AS INTEGER)   AS remaining_months_to_legal_maturity,
                COALESCE(TRY_CAST(p.current_actual_upb AS DOUBLE), 0.0) AS upb_t,
                TRY_CAST(p.current_interest_rate AS DOUBLE)             AS current_interest_rate,

                CASE WHEN p.current_loan_delinquency_status = '1' THEN 1 ELSE 0 END
                    AS is_30dpd_at_t,

                -- modification_flag IN ('Y','P'): Y fires the settlement month only;
                -- P fires all subsequent months.
                CASE WHEN p.modification_flag IN ('Y','P') THEN 1 ELSE 0 END
                    AS modification_flag,

                -- payment_deferral_flag: same Y/P structure as modification_flag.
                -- Distinct from modification_flag since Release 25 (Sept 2020).
                CASE WHEN p.payment_deferral_flag IN ('Y','P') THEN 1 ELSE 0 END
                    AS payment_deferral_flag,

                -- in_workout_plan_flag: borrower_assistance_status_code IS NOT NULL.
                CASE WHEN p.borrower_assistance_status_code IS NOT NULL THEN 1 ELSE 0 END
                    AS in_workout_plan_flag,

                -- deferred_upb_ratio: fraction of outstanding balance that is non-interest-bearing.
                CASE
                    WHEN COALESCE(TRY_CAST(p.current_actual_upb AS DOUBLE), 0.0) > 0
                    THEN COALESCE(TRY_CAST(p.current_non_interest_bearing_upb AS DOUBLE), 0.0)
                         / TRY_CAST(p.current_actual_upb AS DOUBLE)
                    ELSE 0.0
                END AS deferred_upb_ratio

            FROM sf_performance_by_period p
            WHERE p.reporting_year = {obs_year}
              AND p.zero_balance_code IS NULL
              AND p.loan_age >= 1
              AND p.current_loan_delinquency_status IN ('0', '00', '1')
        ),

        -- 3-month prior UPB lookup (for upb_rel_change_3m computation)
        -- Reads two partition years to cover month t-3 near year boundaries
        upb3m AS (
            SELECT
                loan_sequence_number,
                monthly_reporting_period + INTERVAL '3' MONTH AS obs_month,
                TRY_CAST(current_actual_upb AS DOUBLE) AS upb_3m_ago
            FROM sf_performance_by_period
            WHERE reporting_year IN ({y_prev}, {obs_year})
        )

        SELECT
            b.loan_sequence_number,
            b.obs_month,
            YEAR(b.obs_month) AS obs_year,

            -- Target: 60+ DPD or REO Acquisition in (t, t+12].
            COALESCE(lbl.y, 0) AS y,

            -- Dynamic features at observation time t
            b.is_30dpd_at_t,
            b.loan_age,
            b.remaining_months_to_legal_maturity,
            b.upb_t AS current_upb,
            b.current_interest_rate,
            b.modification_flag,
            b.payment_deferral_flag,
            b.in_workout_plan_flag,
            b.deferred_upb_ratio,

            -- Pre-aggregated lagged delinquency features over [t-12, t-1]
            COALESCE(lag.n_times_30dpd_last_12m, 0)   AS n_times_30dpd_last_12m,
            COALESCE(lag.max_dlq_last_12m, 0)          AS max_dlq_last_12m,
            COALESCE(lag.months_since_last_dlq, 999)   AS months_since_last_dlq,

            -- upb_rel_change_3m: (UPB_t - UPB_t_minus_3) / UPB_t_minus_3, a signed 3-month relative UPB change.
            -- Populated only when outside the origination masking window AND not in modified states,
            -- so post-modification balance jumps do not proxy for the excluded modification variables.
            -- The loan_age >= 10 guard ensures t-3 also clears the 6-month masking window:
            -- at obs_month with loan_age L, the t-3 record has loan_age L-3; requiring L-3 > 6
            -- gives L >= 10.
            CASE
                WHEN b.loan_age >= 10
                 AND b.modification_flag = 0
                 AND COALESCE(u.upb_3m_ago, 0.0) > 0.0
                THEN (b.upb_t - u.upb_3m_ago) / u.upb_3m_ago
                ELSE NULL
            END AS upb_rel_change_3m,

            -- amortization_ratio: populated only when loan_age >= 7 (outside masking window).
            CASE
                WHEN b.loan_age >= 7 AND o.original_upb > 0.0
                THEN b.upb_t / CAST(o.original_upb AS DOUBLE)
                ELSE NULL
            END AS amortization_ratio,

            -- rate_spread: current rate vs original rate; reflects rate modification effect.
            CASE
                WHEN o.original_interest_rate IS NULL OR b.current_interest_rate IS NULL THEN NULL
                ELSE b.current_interest_rate - o.original_interest_rate
            END AS rate_spread,

            -- Static origination features
            o.vintage_year,
            o.vintage_quarter,
            o.origination_rule_group,
            o.credit_score,
            o.fico_missing,
            o.original_ltv,
            o.ltv_missing,
            o.original_cltv,
            o.cltv_missing,
            o.original_dti,
            o.dti_missing,
            o.dti_missing_relief_refi,
            o.dti_missing_non_relief_refi,
            o.original_interest_rate,
            o.original_upb,
            o.original_loan_term,
            o.loan_purpose,
            o.channel,
            o.occupancy_status,
            o.property_type,
            o.number_of_units,
            o.is_multi_borrower,
            o.first_time_homebuyer_flag,
            o.property_state,
            o.census_division,
            o.relief_refi_flag,
            o.harp_flag

        -- precomp_prepay is intentionally NOT joined here: the prepayment forward-window
        -- flag is a standalone competing-risk diagnostic read directly from its own parquet
        -- by NB06, not a panel/model feature. Do not add it to the panel feature set.
        FROM base b
        LEFT JOIN  precomp_label  lbl ON b.loan_sequence_number = lbl.loan_sequence_number AND b.obs_month = lbl.obs_month
        LEFT JOIN  precomp_lagged lag ON b.loan_sequence_number = lag.loan_sequence_number AND b.obs_month = lag.obs_month
        LEFT JOIN  upb3m u ON b.loan_sequence_number = u.loan_sequence_number AND b.obs_month = u.obs_month
        INNER JOIN precomp_orig   o   ON b.loan_sequence_number = o.loan_sequence_number
    )
    TO '{out_file.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 500000)
    """)

    elapsed = time.time() - t0
    r = con.execute(
        f"SELECT COUNT(*) AS n, CAST(SUM(y) AS BIGINT) AS pos "
        f"FROM read_parquet('{out_file.as_posix()}')"
    ).fetchone()

    return {
        "year": obs_year, "n_rows": r[0], "n_positive": r[1],
        "positive_rate_pct": round(r[1] / r[0] * 100, 4) if r[0] > 0 else 0.0,
        "status": "computed", "elapsed_s": elapsed,
    }

print("✓  build_panel_year() defined.")

✓  build_panel_year() defined.


In [ ]:
# ── Execute the year loop ─────────────────────────────────────────────────────
# Idempotent by default, but FORCE_REBUILD_PANEL=True clears cached panel outputs
# so that split changes / feature-policy changes are guaranteed to propagate.

if FORCE_REBUILD_PANEL:
    paths_to_clear = [
        PANEL_TEMP_DIR,
        PANEL_FINAL_DIR,
        DRIVE_ROOT / "data" / "panel_temp",
        DRIVE_PANEL,
    ]

    for p in paths_to_clear:
        if p.exists():
            shutil.rmtree(p)
            print(f"⚠  FORCE_REBUILD_PANEL=True - deleted cached panel artifact: {p}")
        p.mkdir(parents=True, exist_ok=True)

year_results = []
print(f"Processing {len(PROCESS_YEARS)} years: {PROCESS_YEARS[0]}-{PROCESS_YEARS[-1]}")
print(f"Output dir: {PANEL_TEMP_DIR}\n")

total_start = time.time()

for obs_year in tqdm(PROCESS_YEARS, desc="Panel assembly", unit="year"):
    try:
        result = build_panel_year(
            con=con,
            obs_year=obs_year,
            out_dir=PANEL_TEMP_DIR,
            skip_existing=not FORCE_REBUILD_PANEL,
        )
        year_results.append(result)
        flag  = "⏭" if result["status"] == "skipped" else "✓"
        t_str = f"{result.get('elapsed_s',0):.0f}s" if result["status"] == "computed" else "(cached)"
        print(f"  {flag} {obs_year}: {result['n_rows']:>12,} rows  pos={result['positive_rate_pct']:.3f}%  {t_str}")
    except Exception as exc:
        print(f"  ✗  {obs_year}: FAILED - {exc}")
        year_results.append({"year": obs_year, "status": "error", "error": str(exc)})

print(f"\n{'─'*60}")

error_results = [r for r in year_results if r.get("status") == "error"]
if error_results:
    preview = "; ".join(
        f"{r['year']}: {r['error']}" for r in error_results[:5]
    )
    more = "" if len(error_results) <= 5 else f"; ... ({len(error_results)} years failed total)"
    raise RuntimeError(
        "Year loop failed before producing complete panel_temp output. "
        f"First failures: {preview}{more}"
    )

results_df = pd.DataFrame([r for r in year_results if "n_rows" in r])
if len(results_df):
    overall_pos = results_df["n_positive"].sum() / results_df["n_rows"].sum() * 100
    print(f"Total rows       : {results_df['n_rows'].sum():,}")
    print(f"Total positive   : {results_df['n_positive'].sum():,}")
    print(f"Overall pos rate : {overall_pos:.3f}%")
    n_errors = sum(1 for r in year_results if r.get("status") == "error")
    if n_errors:
        print(f"⚠  {n_errors} year(s) FAILED - re-run to retry.")
    else:
        print("✓  All years completed.")

Processing 25 years: 1999–2023
Output dir: /content/panel_temp



Panel assembly:   0%|          | 0/25 [00:00<?, ?year/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  ✓ 1999:    6,333,772 rows  pos=0.701%  3s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

  ✓ 2021:  136,071,433 rows  pos=0.952%  26s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  ✓ 2022:  151,205,691 rows  pos=0.991%  29s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  ✓ 2023:  155,135,189 rows  pos=1.093%  26s

────────────────────────────────────────────────────────────
Total rows       : 2,408,959,003
Total positive   : 41,725,929
Overall pos rate : 1.732%
✓  All years completed.


#### Drive Sync - Panel Year Files

In [ ]:
if IS_COLAB:
    drive_temp = DRIVE_ROOT / "data" / "panel_temp"
    drive_temp.mkdir(parents=True, exist_ok=True)
    missing_on_drive = []
    for obs_year in PROCESS_YEARS:
        local_f = PANEL_TEMP_DIR / f"obs_year={obs_year}" / "panel.parquet"
        drive_f = drive_temp / f"obs_year={obs_year}" / "panel.parquet"
        if local_f.exists() and not drive_f.exists():
            missing_on_drive.append((local_f, drive_f))
    if missing_on_drive:
        print(f"Syncing {len(missing_on_drive)} year file(s) to Drive...")
        t0 = time.time()
        for local_f, drive_f in missing_on_drive:
            drive_f.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(local_f, drive_f)
        print(f"✓  Synced")
    else:
        print("✓  All year files already on Drive.")

Syncing 25 year file(s) to Drive...
✓  Synced


---
## Section 4 · Post-Processing: Split Labels and Subgroup Assignments

Combines the year-level panel files into the final split-partitioned panel.

Three tasks:

1. Assigns each row to the canonical chronological split from `SPLIT_WINDOWS`.
2. Constructs subgroup variables used for later coverage diagnostics and Mondrian CP calibration.
3. Writes the final partitioned Parquet panel.

Splits are chronological rather than random because the data are temporally ordered loan-month observations (see §0.6).

Subgroup assignments are used for calibration/evaluation, not as a claim of causal structure. The FICO × LTV grouping is economically motivated: borrower credit quality and leverage / equity position are standard dimensions of mortgage-default risk, with LTV and negative equity playing a central role in default incentives and mortgage-risk prediction (Campbell & Cocco, 2015; Foote et al., 2008; Fitzpatrick & Mues, 2016).

- `fico_tier` is the implementation label for thesis-defined tiers of the observed SFLLD `CREDIT SCORE`; missing credit score is assigned to `Sentinel` rather than imputed. The 640 and 720 cutoffs are an explicit analytical choice for this thesis and are applied identically here and in NB02's subgroup analysis. These cutoffs define evaluation/calibration subgroups only and are not model inputs. Freddie Mac's supplied documentation names the source field `CREDIT SCORE` and does not identify its third-party scoring model (Freddie Mac, 2026b).
- `ltv_bucket` is based on observed origination LTV only; missing or disclosure-invalid LTV is assigned to `Sentinel`.
- `fico_ltv_group` is the cross-product subgroup later used for Mondrian CP cell diagnostics.
- `census_division` follows official Census divisions for states/DC and pragmatic assignments for territories. It is the one subgroup field that is also an active model feature: it enters the model as a string categorical and is additionally used as a Mondrian CP subgroup dimension.

The `fico_tier`, `ltv_bucket`, and `fico_ltv_group` subgroup fields are used only for evaluation and calibration and are not model inputs (the model receives `credit_score` and `original_ltv` directly). `census_division` is the exception: it is both an active model feature and a subgroup dimension, as noted above.


In [ ]:
# Verify all year files present before post-processing
missing_years = [
    y for y in PROCESS_YEARS
    if not (PANEL_TEMP_DIR / f"obs_year={y}" / "panel.parquet").exists()
]
if not missing_years:
    print(f"✓  All {len(PROCESS_YEARS)} year files present on local NVMe.")
else:
    print(f"⚠  {len(missing_years)} year file(s) not found on local NVMe "
          f"(first year missing: {missing_years[0]}). Will raise if assembly is needed.")

# temp_glob: Hive-style path for the year-partitioned temp files.
temp_glob = (PANEL_TEMP_DIR / "obs_year=*" / "panel.parquet").as_posix()

✓  All 25 year files present on local NVMe.


In [ ]:
# Idempotency check - skip if all split partitions already exist,
# unless FORCE_REBUILD_PANEL=True requests a clean rebuild.

if FORCE_REBUILD_PANEL and PANEL_FINAL_DIR.exists():
    shutil.rmtree(PANEL_FINAL_DIR)
    PANEL_FINAL_DIR.mkdir(parents=True, exist_ok=True)
    print("⚠  FORCE_REBUILD_PANEL=True - deleted cached final panel partitions for a clean rebuild.")

expected_splits = list(SPLIT_WINDOWS.keys())
existing_partitions = [
    s for s in expected_splits
    if len(list((DRIVE_PANEL / f"split={s}").glob("*.parquet"))) > 0
]

if set(existing_partitions) == set(expected_splits):
    total_rows = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{(DRIVE_PANEL / '**' / '*.parquet').as_posix()}', union_by_name=True)"
    ).fetchone()[0]
    print(f"⏭  Final panel already assembled ({total_rows:,} rows). Skipping.")
else:
    # Year files are required for assembly
    if missing_years:
        raise RuntimeError(
            f"Missing year outputs: {missing_years}. Re-run §3.6 loop."
        )
    missing = set(expected_splits) - set(existing_partitions)
    print(f"Assembling final panel  →  {PANEL_FINAL_DIR}")
    print(f"  Missing partitions: {sorted(missing)}")
    t0 = time.time()

    split_case = "\n        ".join(
        f"WHEN obs_month BETWEEN DATE '{start}' AND DATE '{end}' THEN '{name}'"
        for name, (start, end) in SPLIT_WINDOWS.items()
    )

    con.execute(f"""
    COPY (
        -- CTE labels every row and assigns subgroups; the outer SELECT filters out
        -- buffer rows so they are never written to the final panel partitions.
        WITH labeled AS (
            SELECT
                *,

                -- ── Temporal split label ──────────────────────────────────────────
                CASE
                {split_case}
                ELSE 'buffer'
                END AS split,

                -- ── Credit-score tier (`fico_tier` implementation name) for Mondrian CP subgrouping ─────────────────────────
                CASE
                    WHEN fico_missing = 1 THEN 'Sentinel'
                    WHEN credit_score < 640 THEN 'Subprime'
                    WHEN credit_score < 720 THEN 'Near-Prime'
                    ELSE                        'Prime'
                END AS fico_tier,

                -- ── LTV bucket for Mondrian CP subgrouping ────────────────────────
                CASE
                    WHEN ltv_missing = 1    THEN 'Sentinel'
                    WHEN original_ltv <= 80 THEN 'Low'
                    WHEN original_ltv <= 95 THEN 'Moderate'
                    ELSE                        'High'
                END AS ltv_bucket,

                -- ── Credit-score × LTV cross-product (`fico_ltv_group` implementation name) ──────────────────────────────────────
                CASE
                    WHEN fico_missing = 1 OR ltv_missing = 1 THEN 'Sentinel'
                    ELSE
                        CASE WHEN credit_score < 640 THEN 'Subprime'
                             WHEN credit_score < 720 THEN 'Near-Prime'
                             ELSE                        'Prime' END
                        || ' × ' ||
                        CASE WHEN original_ltv <= 80 THEN 'Low'
                             WHEN original_ltv <= 95 THEN 'Moderate'
                             ELSE                         'High' END
                END AS fico_ltv_group

            FROM read_parquet('{temp_glob}', union_by_name=True)
        )
        SELECT * FROM labeled WHERE split != 'buffer'
    )
    TO '{PANEL_FINAL_DIR.as_posix()}'
    (FORMAT PARQUET, PARTITION_BY (split), COMPRESSION ZSTD,
     ROW_GROUP_SIZE 500000, OVERWRITE_OR_IGNORE TRUE)
    """)

    elapsed = time.time() - t0
    total_rows = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{(PANEL_FINAL_DIR / '**' / '*.parquet').as_posix()}', union_by_name=True)"
    ).fetchone()[0]
    print(f"✓  Final panel assembled ({total_rows:,} rows)")

Assembling final panel  →  /content/panel_final
  Missing partitions: ['calibration', 'test_covid', 'test_normal', 'test_rate_hike', 'test_subprime', 'train']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓  Final panel assembled (2,322,200,609 rows)


#### Drive Sync - Final Panel

In [ ]:
# ── Move final panel splits to the canonical Drive location, then clean up ────
# Runs in every environment so the panel and its manifest always live under
# DRIVE_PANEL (DRIVE_ROOT / "data" / "panel"), which NB04 reads.
if IS_COLAB and PANEL_TEMP_DIR.exists():
    print("Deleting local panel_temp to free NVMe space...")
    shutil.rmtree(PANEL_TEMP_DIR)
    print("✓  panel_temp deleted")

missing_splits_drive = [
    s for s in expected_splits
    if not list((DRIVE_PANEL / f"split={s}").glob("*.parquet"))
]

if missing_splits_drive:
    print(f"Moving {len(missing_splits_drive)} split(s) to {DRIVE_PANEL} ...")
    for split_name in missing_splits_drive:
        src_dir  = PANEL_FINAL_DIR / f"split={split_name}"
        dest_dir = DRIVE_PANEL     / f"split={split_name}"
        dest_dir.mkdir(parents=True, exist_ok=True)
        t0 = time.time()
        for f in src_dir.glob("*.parquet"):
            shutil.move(str(f), str(dest_dir / f.name))
        elapsed = time.time() - t0
        if src_dir.exists():
            shutil.rmtree(src_dir)
        size_mb = sum(f.stat().st_size for f in dest_dir.glob("*.parquet")) / 1e6
        print(f"  ✓  {split_name:<18}: {size_mb:.0f} MB in {elapsed:.0f}s")
    print(f"✓  All splits at {DRIVE_PANEL}")
else:
    print(f"✓  All panel splits already at {DRIVE_PANEL}.")

if PANEL_FINAL_DIR.exists():
    shutil.rmtree(PANEL_FINAL_DIR, ignore_errors=True)

# Re-register panel view from the canonical Drive location
panel_glob = (DRIVE_PANEL / "**" / "*.parquet").as_posix()
con.execute(
    f"CREATE OR REPLACE VIEW panel AS "
    f"SELECT * FROM read_parquet('{panel_glob}', union_by_name=True)"
)
total_panel = con.execute("SELECT COUNT(*) FROM panel").fetchone()[0]
print(f"✓  Panel view re-registered ({total_panel:,} rows) from: {DRIVE_PANEL}")

Deleting local panel_temp to free NVMe space...
✓  panel_temp deleted
Moving 6 split(s) to /content/drive/MyDrive/master_thesis/data/panel ...
  ✓  train             : 6380 MB in 37s
  ✓  calibration       : 5234 MB in 29s
  ✓  test_subprime     : 18825 MB in 112s
  ✓  test_normal       : 23567 MB in 142s
  ✓  test_covid        : 5237 MB in 29s
  ✓  test_rate_hike    : 6506 MB in 35s
✓  All splits at /content/drive/MyDrive/master_thesis/data/panel
✓  Panel view re-registered (2,322,200,609 rows) from: /content/drive/MyDrive/master_thesis/data/panel


---
## Section 5 · Validation Checks

Ten validation checks run before the manifest is written, spanning class balance, forward-window contamination, subgroup behavior, leakage, structural integrity, label completeness, and intra-split stability.

1. **Split-level class balance** - positive rates by split with sanity assertions.
2. **Calibration forward-window contamination** - fraction of calibration rows whose 12-month label window overlaps 2007, with positive-rate bias quantified.
3. **Subgroup monotonicity** - FICO and LTV orderings across splits.
4. **Mondrian CP calibration cell sizes** - practical subgroup-size screen for later Mondrian CP calibration; the size threshold is heuristic, not a formal validity theorem (Vovk, 2012).
5. **Feature-target correlation** - leakage screen for suspicious correlations; it can detect obvious leakage but cannot prove absence of leakage.
6. **Row counts, schema, NULL rates, hard exclusions, and training-window feature availability** - structural integrity diagnostics and active-feature eligibility.
7. **Base-to-panel loan reduction** - reconciles the distinct-loan reduction to the excluded buffer periods; NB02 separately verifies origination-join integrity.
8. **Duplicate observation check** - no duplicate `(loan_sequence_number, obs_month)` pairs.
9. **Fast-default label gap** - quantifies credit-event terminations (`ZBC ∈ {02, 03}`) that never showed an active 60+ DPD / REO-status record.
10. **Within-split positive-rate trend** - checks whether positive rates are stable or drifting within each split, which matters for CP interpretation under non-exchangeability.

Two additional (already stated before) rules apply before training-window variability is checked:

- some fields are hard-excluded from model eligibility because their semantics are not stable enough for the main frozen historical-training model (`channel`, modification / deferral indicators, modified-loan time variables, later-regime policy features);
- origination disclosure sentinels are handled via NULL preservation, so missingness rates for FICO / LTV / CLTV / DTI are substantively meaningful and must be inspected.

The final manifest therefore exposes only features that are:
1. not hard-excluded by design, and
2. variable in the training window.

### Check 1 · Positive Rate by Split

In [ ]:
pos_by_split = con.execute("""
SELECT split,
       COUNT(*) AS n_rows,
       COUNT(DISTINCT loan_sequence_number) AS n_distinct_loans,
       CAST(SUM(y) AS BIGINT) AS n_positive,
       ROUND(100.0 * AVG(y), 4) AS positive_rate_pct,
       MIN(obs_month) AS obs_start,
       MAX(obs_month) AS obs_end
FROM panel
GROUP BY split ORDER BY obs_start
""").df()

print("=== Positive Rate by Split (target: 60+ DPD or REO Acquisition) ===")
print(pos_by_split.to_string(index=False))

split_rates = dict(zip(pos_by_split["split"], pos_by_split["positive_rate_pct"]))
print()
checks = [
    ("test_subprime > calibration (crisis peak > pre-crisis base)",
     split_rates.get("test_subprime", 0) > split_rates.get("calibration", 0)),
    ("test_covid rate context: COVID forbearance suppresses foreclosure progression "
     "(DPD may stay elevated but REO rate is suppressed)",
     True),  # informational only - not a pass/fail assertion
    ("train positive rate > 0.5% (heuristic sanity threshold)",
     split_rates.get("train", 0) > 0.5),
    ("calibration positive rate > 0% (calibration set has positive class support)",
     split_rates.get("calibration", 0) > 0),
]
for desc, passed in checks:
    print(f"  {'✓' if passed else '⚠ CHECK'}  {desc}")

print("\nNote: test_covid positive rate reflects COVID forbearance period during which")
print("  REO progression was suppressed by policy. The label semantics (60+ DPD OR REO)")
print("  mean some COVID loans show high DPD but delayed REO resolution.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Positive Rate by Split (target: 60+ DPD or REO Acquisition) ===
         split    n_rows  n_distinct_loans  n_positive  positive_rate_pct  obs_start    obs_end
         train 248091644          12653155     2867880             1.1560 1999-01-01 2004-12-01
   calibration 179344121           9742593     1986205             1.1075 2005-01-01 2006-12-01
 test_subprime 633244002          17677746    15990464             2.5252 2007-01-01 2012-12-01
   test_normal 753732904          18595092    11436593             1.5173 2013-01-01 2019-09-01
    test_covid 201447058          16570540     3382830             1.6793 2020-03-01 2021-09-01
test_rate_hike 306340880          14958763     3194065             1.0427 2022-01-01 2023-12-01

  ✓  test_subprime > calibration (crisis peak > pre-crisis base)
  ✓  test_covid rate context: COVID forbearance suppresses foreclosure progression (DPD may stay elevated but REO rate is suppressed)
  ✓  train positive rate > 0.5% (minimum plausibility for gr

### Check 2 · Calibration Forward-Window Contamination

Calibration ends at 2006-12-31, but the 12-month forward label window for observations in 2006 can extend into 2007. Because the target is defined over `(t, t+12]`, this is a mechanical consequence of the label horizon.

This matters because 2007 begins the explicit `test_subprime` regime. Positive events inside 2007 can therefore affect labels for late-calibration observation months. Forward-window overlap can move the calibration positive rate in either direction relative to a purely 2005-2006 event window (it depends on whether late-2006 observation months happen to carry higher or lower forward-event rates than early-calibration months). The check below quantifies the realised direction and magnitude at the positive-rate level. The effect on the conformal threshold $\hat{q}$ is a separate score-level question:
$\hat{q}$ is a quantile of the nonconformity-score distribution, not of the positive
rate, and no clean-versus-overlap threshold counterfactual is run downstream.

A calibration observation at month `t` is treated as forward-window contaminated when its label window `(t, t+12]` reaches into 2007, i.e. `t >= 2006-01-01`: an observation at January 2006 has label window `(2006-01, 2007-01]`, which already includes January 2007. All calibration rows from January 2006 onward therefore have label windows that reach into 2007.


In [ ]:
CAL_CONTAMINATION_CUTOFF = '2006-01-01'  # obs_month >= this date has label window in 2007+

cal_total = con.execute(
    "SELECT COUNT(*) FROM panel WHERE split = 'calibration'"
).fetchone()[0]

cal_contaminated = con.execute(f"""
    SELECT COUNT(*) FROM panel
    WHERE split = 'calibration'
      AND obs_month >= DATE '{CAL_CONTAMINATION_CUTOFF}'
""").fetchone()[0]

cal_contam_pct = 100.0 * cal_contaminated / cal_total if cal_total else 0.0

cal_pos_clean = con.execute(f"""
    SELECT ROUND(100.0 * AVG(y), 4)
    FROM panel
    WHERE split = 'calibration'
      AND obs_month < DATE '{CAL_CONTAMINATION_CUTOFF}'
""").fetchone()[0]

cal_pos_contam = con.execute(f"""
    SELECT ROUND(100.0 * AVG(y), 4)
    FROM panel
    WHERE split = 'calibration'
      AND obs_month >= DATE '{CAL_CONTAMINATION_CUTOFF}'
""").fetchone()[0]

print("=== Calibration Forward-Window Contamination ===")
print(f"  Total calibration rows      : {cal_total:>12,}")
print(f"  Contaminated rows (obs >= {CAL_CONTAMINATION_CUTOFF}): {cal_contaminated:>12,}  ({cal_contam_pct:.1f}%)")
print(f"  Positive rate - clean half  : {cal_pos_clean:.4f}%")
print(f"  Positive rate - contam half : {cal_pos_contam:.4f}%")
print()

# Estimate the overall calibration positive rate and clean-half counterfactual.
# This quantifies the rate effect only; it does not identify the effect on q̂.
cal_pos_overall = con.execute(
    "SELECT ROUND(100.0 * AVG(y), 4) FROM panel WHERE split = 'calibration'"
).fetchone()[0]

# Counterfactual: what would the overall calibration positive rate be if the
# contaminated rows had the same positive rate as the clean half?
if cal_total > 0 and cal_pos_clean is not None and cal_pos_contam is not None:
    # Overall rate = (clean_n / total) * clean_rate + (contam_n / total) * contam_rate
    # Counterfactual: clean-half rate applied throughout. Since the contaminated half is
    # replaced by the clean rate, the counterfactual overall rate equals the clean rate.
    counterfactual_pos_rate = float(cal_pos_clean)
    bias_pp = float(cal_pos_overall) - counterfactual_pos_rate

    if bias_pp > 0:
        direction_word = "raises"
    elif bias_pp < 0:
        direction_word = "lowers"
    else:
        direction_word = "leaves unchanged"

    print(f"  Overall calibration positive rate                        : {cal_pos_overall:.4f}%")
    print(f"  Counterfactual rate (clean-half rate applied throughout) : {counterfactual_pos_rate:.4f}%")
    print(f"  Contamination effect on positive rate                    : {bias_pp:+.4f} pp")
    print()
    print("Interpretation:")
    print(f"  Relative to the clean-half counterfactual, forward-window contamination "
          f"{direction_word}")
    print(f"  the calibration positive rate by {abs(bias_pp):.4f} pp in this build.")
    print()
    print("CP threshold note:")
    print("  This positive-rate comparison does NOT determine the direction of the effect on the")
    print("  conformal threshold q̂: q̂ is the (1-α) quantile of the nonconformity-score")
    print("  distribution, which depends on the joint distribution of model scores and labels in")
    print("  the contaminated half, not on the positive rate alone. The score-level effect is")


CALIBRATION_CONTAM = {
    "cutoff"                    : CAL_CONTAMINATION_CUTOFF,
    "n_total_rows"              : int(cal_total),
    "n_contaminated_rows"       : int(cal_contaminated),
    "contaminated_share_pct"    : round(cal_contam_pct, 4),
    "pos_rate_clean_pct"        : float(cal_pos_clean),
    "pos_rate_contaminated_pct" : float(cal_pos_contam),
    "delta_pp"                  : round(bias_pp, 4),
}

=== Calibration Forward-Window Contamination ===
  Total calibration rows      :  179,344,121
  Contaminated rows (obs >= 2006-01-01):   92,861,653  (51.8%)
  Positive rate - clean half  : 1.1258%
  Positive rate - contam half : 1.0904%

  Overall calibration positive rate                        : 1.1075%
  Counterfactual rate (clean-half rate applied throughout) : 1.1258%
  Contamination effect on positive rate                    : -0.0183 pp

Interpretation:
  Relative to the clean-half counterfactual, forward-window contamination lowers
  the calibration positive rate by 0.0183 pp in this build.

CP threshold note:
  This positive-rate comparison does NOT determine the direction of the effect on the
  conformal threshold q̂: q̂ is the (1-α) quantile of the nonconformity-score
  distribution, which depends on the joint distribution of model scores and labels in
  the contaminated half, not on the positive rate alone. The score-level effect is
  quantified empirically where the thresh

### Check 2b · test_normal COVID Forward-Window Contamination

`test_normal` ends 2019-09-30 to remove observation months whose forward windows are
dominated by COVID-era events, but trimming the observation window cannot remove the
overlap entirely: any observation month `t` whose 12-month label window `(t, t+12]`
reaches the start of the COVID regime carries a label partially determined by
COVID-era events. With the COVID regime starting at the `test_covid` split boundary,
this affects all `test_normal` observation months from twelve months before that
boundary onward.

This check quantifies (i) the share of `test_normal` rows affected, and (ii) the
positive-rate difference between the affected and unaffected portions, mirroring
Check 2. The computed quantities are written into `panel_manifest.json`
(`forward_window_contamination`) so downstream regime interpretation (NB06/NB07)
references measured values instead of re-deriving or asserting them.

In [ ]:
# An observation month t has label window (t, t+12] intersecting the COVID regime
# if t + 12 months >= COVID_REGIME_START, i.e. obs_month >= COVID_REGIME_START - 12 months.
COVID_REGIME_START = SPLIT_WINDOWS["test_covid"][0]

tn = con.execute(f"""
SELECT
    COUNT(*)                                                                          AS n_total,
    COUNT(*) FILTER (WHERE obs_month >= DATE '{COVID_REGIME_START}' - INTERVAL '12' MONTH) AS n_contam,
    ROUND(100.0 * AVG(y) FILTER (WHERE obs_month <  DATE '{COVID_REGIME_START}' - INTERVAL '12' MONTH), 4) AS pos_clean,
    ROUND(100.0 * AVG(y) FILTER (WHERE obs_month >= DATE '{COVID_REGIME_START}' - INTERVAL '12' MONTH), 4) AS pos_contam,
    ROUND(100.0 * AVG(y), 4)                                                          AS pos_overall
FROM panel
WHERE split = 'test_normal'
""").fetchone()

tn_total, tn_contam, tn_pos_clean, tn_pos_contam, tn_pos_overall = tn
tn_contam_pct = 100.0 * tn_contam / tn_total if tn_total else 0.0
tn_delta_pp   = float(tn_pos_contam) - float(tn_pos_clean)

if tn_delta_pp > 0:
    tn_direction = "higher in"
elif tn_delta_pp < 0:
    tn_direction = "lower in"
else:
    tn_direction = "equal between"

print("=== test_normal COVID Forward-Window Contamination ===")
print(f"  COVID regime start (test_covid boundary)     : {COVID_REGIME_START}")
print(f"  Affected obs_months                           : obs_month >= {COVID_REGIME_START} - 12 months")
print(f"  Total test_normal rows                        : {tn_total:>12,}")
print(f"  Rows with COVID-overlapping label windows     : {tn_contam:>12,}  ({tn_contam_pct:.1f}%)")
print(f"  Positive rate - unaffected portion            : {tn_pos_clean:.4f}%")
print(f"  Positive rate - affected portion              : {tn_pos_contam:.4f}%")
print(f"  Difference (affected - unaffected)            : {tn_delta_pp:+.4f} pp")
print()
print("Interpretation:")
print(f"  The positive rate is {tn_direction} the COVID-overlapping portion of test_normal.")
print("  Labels in the affected portion are partially determined by COVID-regime events;")
print("  downstream regime characterisations of test_normal (NB06/NB07) must condition on")
print("  the affected-share and rate-difference figures recorded in panel_manifest.json")
print("  rather than treating the split as homogeneous.")

TEST_NORMAL_COVID_CONTAM = {
    "covid_regime_start"        : str(COVID_REGIME_START),
    "n_total_rows"              : int(tn_total),
    "n_contaminated_rows"       : int(tn_contam),
    "contaminated_share_pct"    : round(tn_contam_pct, 4),
    "pos_rate_clean_pct"        : float(tn_pos_clean),
    "pos_rate_contaminated_pct" : float(tn_pos_contam),
    "delta_pp"                  : round(tn_delta_pp, 4),
}

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== test_normal COVID Forward-Window Contamination ===
  COVID regime start (test_covid boundary)     : 2020-03-01
  Affected obs_months                           : obs_month >= 2020-03-01 - 12 months
  Total test_normal rows                        :  753,732,904
  Rows with COVID-overlapping label windows     :   70,201,657  (9.3%)
  Positive rate - unaffected portion            : 1.3423%
  Positive rate - affected portion              : 3.2211%
  Difference (affected - unaffected)            : +1.8788 pp

Interpretation:
  The positive rate is higher in the COVID-overlapping portion of test_normal.
  Labels in the affected portion are partially determined by COVID-regime events;
  downstream regime characterisations of test_normal (NB06/NB07) must condition on
  the affected-share and rate-difference figures recorded in panel_manifest.json
  rather than treating the split as homogeneous.


### Check 3 · Subgroup Positive Rate Monotonicity


In [ ]:
print("=== FICO monotonicity (Subprime > Near-Prime > Prime) ===")
# Restrict this QC heuristic to train/calibration/pre-pandemic test windows.
# COVID is policy-distorted, while the rate-hike window has a distinct
# composition-shift interpretation; neither exclusion affects downstream evaluation.
for split_name in ["train", "calibration", "test_subprime", "test_normal"]:
    fico = con.execute(f"""
        SELECT fico_tier, ROUND(100.0 * AVG(y), 4) AS pr
        FROM panel WHERE split='{split_name}' AND fico_tier != 'Sentinel'
        GROUP BY 1
    """).df().set_index("fico_tier")["pr"].to_dict()
    ok = fico.get("Subprime",0) > fico.get("Near-Prime",0) > fico.get("Prime",0)
    print(f"  {'✓' if ok else '⚠'} {split_name}: "
          f"Sub={fico.get('Subprime',0):.3f}% > "
          f"Near={fico.get('Near-Prime',0):.3f}% > "
          f"Prime={fico.get('Prime',0):.3f}%")

print("\n=== LTV monotonicity (Low < Moderate < High) ===")
# Same diagnostic scope as the FICO check above.
for split_name in ["train", "calibration", "test_subprime", "test_normal"]:
    ltv = con.execute(f"""
        SELECT ltv_bucket, ROUND(100.0 * AVG(y), 4) AS pr
        FROM panel WHERE split='{split_name}' AND ltv_bucket != 'Sentinel'
        GROUP BY 1
    """).df().set_index("ltv_bucket")["pr"].to_dict()
    # Default 0 for all buckets: a missing bucket silently fails the comparison
    # (0 < 0 is False). This is consistent with the FICO check defaults above.
    ok = ltv.get("Low", 0) < ltv.get("Moderate", 0) < ltv.get("High", 0)
    print(f"  {'✓' if ok else '⚠'} {split_name}: "
          f"Low={ltv.get('Low',0):.3f}% < "
          f"Mod={ltv.get('Moderate',0):.3f}% < "
          f"High={ltv.get('High',0):.3f}%")

=== FICO monotonicity (Subprime > Near-Prime > Prime) ===
  ✓ train: Sub=5.996% > Near=1.581% > Prime=0.252%


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  ✓ calibration: Sub=5.746% > Near=1.735% > Prime=0.299%
  ✓ test_subprime: Sub=10.200% > Near=4.516% > Prime=1.021%


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  ✓ test_normal: Sub=7.138% > Near=3.066% > Prime=0.708%

=== LTV monotonicity (Low < Moderate < High) ===
  ✓ train: Low=0.844% < Mod=2.869% < High=3.187%


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  ✓ calibration: Low=0.870% < Mod=2.770% < High=3.667%
  ✓ test_subprime: Low=2.134% < Mod=4.765% < High=5.715%
  ✓ test_normal: Low=1.340% < Mod=1.994% < High=2.234%


### Check 4 · Mondrian CP Calibration Cell Sizes

`MONDRIAN_MIN_CELL` distinct loans is a **pragmatic viability threshold**, not a formal Mondrian CP requirement.

Category-conditional, Mondrian-style conformal calibration uses predefined groups and calibrates within those groups to target group-conditional rather than only marginal coverage. The general Mondrian conformal-prediction construction is treated in Vovk et al. (2005), while Vovk (2012) discusses conditional-validity notions for inductive conformal predictors. The `MONDRIAN_MIN_CELL` threshold used here is only a practical stability threshold, not a formal validity condition. However, very small calibration groups produce noisy empirical quantiles and unstable empirical coverage estimates. At `alpha = 0.10`, the simple binomial sampling-noise scale is approximately

`sqrt(alpha * (1 - alpha) / MONDRIAN_MIN_CELL) ≈ 1.34 pp` at the current `ALPHA` and `MONDRIAN_MIN_CELL` settings.

This rule is therefore a practical diagnostic threshold: cells below 500 distinct loans are flagged as too sparse for stable subgroup calibration/evaluation in later notebooks. The threshold should not be described as a theorem.

The check counts **distinct loans** rather than loan-month rows because loan-month observations are correlated within a loan. Five hundred loan-months from a small number of loans are less informative than five hundred distinct loans, which is consistent with standard panel-data dependence concerns (Wooldridge, 2010).

In [ ]:
cell_sizes = con.execute("""
SELECT fico_ltv_group,
       COUNT(DISTINCT loan_sequence_number) AS n_distinct_loans,
       COUNT(*) AS n_loan_months,
       ROUND(100.0 * AVG(y), 4) AS positive_rate_pct
FROM panel
WHERE split = 'calibration' AND fico_ltv_group != 'Sentinel'
GROUP BY 1 ORDER BY 1
""").df()

print("=== Calibration FICO×LTV Cell Sizes ===")
print(cell_sizes.to_string(index=False))
too_small = cell_sizes[cell_sizes["n_distinct_loans"] < MONDRIAN_MIN_CELL]
if len(too_small) == 0:
    print(f"\n✓  All cells ≥ {MONDRIAN_MIN_CELL} loans. Smallest: {cell_sizes['n_distinct_loans'].min():,}.")
else:
    print(f"\n⚠  {len(too_small)} cell(s) below {MONDRIAN_MIN_CELL} loans:")
    print(too_small.to_string(index=False))

print("\n=== Census Division Calibration Cell Sizes ===")
div_sizes = con.execute("""
    SELECT census_division, COUNT(DISTINCT loan_sequence_number) AS n_distinct_loans
    FROM panel WHERE split='calibration'
    GROUP BY 1 ORDER BY 2
""").df()
print(div_sizes.to_string(index=False))
small_divs = div_sizes[div_sizes["n_distinct_loans"] < MONDRIAN_MIN_CELL]
if len(small_divs):
    print(f"\n⚠  {len(small_divs)} division(s) below threshold: {small_divs['census_division'].tolist()}")
else:
    print(f"\n✓  All Census divisions viable (≥ {MONDRIAN_MIN_CELL} loans).")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Calibration FICO×LTV Cell Sizes ===
       fico_ltv_group  n_distinct_loans  n_loan_months  positive_rate_pct
    Near-Prime × High             37843         556319             4.2706
     Near-Prime × Low           2645334       47166131             1.4225
Near-Prime × Moderate            542602        9515042             3.1346
         Prime × High             29042         469140             1.6837
          Prime × Low           5298716      100930335             0.2483
     Prime × Moderate            476838        8695106             0.8191
      Subprime × High              8587         103181             9.2856
       Subprime × Low            526427        8848556             5.0089
  Subprime × Moderate            145606        2442791             8.2663

✓  All cells ≥ 500 loans. Smallest: 8,587.

=== Census Division Calibration Cell Sizes ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

census_division  n_distinct_loans
E South Central            452163
    New England            470499
W South Central            713649
       Mountain            805553
W North Central            879450
Middle Atlantic           1089377
        Pacific           1383960
E North Central           1901206
 South Atlantic           2046736

✓  All Census divisions viable (≥ 500 loans).


### Check 5 · Feature-Target Correlation (Leakage Screen)

Computes simple feature-target correlations for a selected set of numeric panel fields as a **leakage screen**. Very high absolute correlations can flag suspicious variables that may encode target information directly or indirectly.

This is not a proof of no leakage. Leakage can be nonlinear, conditional on subgroups, or embedded in temporal ordering. The check is therefore interpreted as a first-pass diagnostic only. The general leakage concern is best understood as a data-design and learn-predict-separation problem rather than as something that can be ruled out by one marginal statistic (Kaufman et al., 2012). General predictive-modeling practice also treats predictor screening and preprocessing as model-development diagnostics rather than formal guarantees (Kuhn & Johnson, 2013).

In [ ]:
# Pearson |r| > 0.30 is treated as a strong pragmatic warning sign.
# This cutoff is not a literature theorem. It is only a screening heuristic:
# low marginal correlation does not prove absence of leakage because leakage
# can be nonlinear, conditional, temporal, or embedded in data construction
# (Kaufman et al., 2012).

# Diagnostic subset only: this is not the exhaustive active-feature list.
# Categorical fields and some numeric active fields are not included in this screen.

numeric_feature_candidates = [
    "is_30dpd_at_t", "loan_age", "current_upb", "current_interest_rate",
    "modification_flag", "payment_deferral_flag", "deferred_upb_ratio",
    "n_times_30dpd_last_12m", "max_dlq_last_12m", "months_since_last_dlq",
    "upb_rel_change_3m", "amortization_ratio", "rate_spread",
    "credit_score", "original_ltv", "original_cltv", "original_dti",
    "fico_missing", "ltv_missing", "cltv_missing",
    "dti_missing", "dti_missing_relief_refi", "dti_missing_non_relief_refi",
    "original_upb", "relief_refi_flag", "harp_flag", "is_multi_borrower",
    "number_of_units",
]

corr_exprs = ",\n    ".join(
    f"ROUND(CORR({f}, y), 4) AS {f}" for f in numeric_feature_candidates
)

corr_row = con.execute(f"""
SELECT {corr_exprs}
FROM panel WHERE split = 'train'
""").df()

corr_df = (
    corr_row.T
    .rename(columns={0: "pearson_r"})
    .sort_values("pearson_r", key=lambda s: s.abs().fillna(-1), ascending=False)
)

corr_df["status"] = [
    "hard_excluded (diagnostic only)" if f in HARD_EXCLUDED_MODEL_FEATURES else "candidate"
    for f in corr_df.index
]

print("=== Feature-Target Pearson Correlation (training split) ===")
print("Note: rows marked 'hard_excluded' are shown for leakage diagnostics only and are")
print("      never eligible model features (see §0.6 HARD_EXCLUDED_MODEL_FEATURES).")
print(corr_df.to_string())

suspicious = corr_df[corr_df["pearson_r"].abs() > 0.30]
if len(suspicious) == 0:
    print("\n✓  No feature shows |r| > 0.30 in the training split.")
else:
    print(f"\n⚠  {len(suspicious)} feature(s) with |r| > 0.30:")
    print(suspicious.to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Feature-Target Pearson Correlation (training split) ===
Note: rows marked 'hard_excluded' are shown for leakage diagnostics only and are
      never eligible model features (see §0.6 HARD_EXCLUDED_MODEL_FEATURES).
                             pearson_r                           status
is_30dpd_at_t                   0.3299                        candidate
n_times_30dpd_last_12m          0.3263                        candidate
max_dlq_last_12m                0.2951                        candidate
months_since_last_dlq          -0.2615                        candidate
credit_score                   -0.1363                        candidate
current_interest_rate           0.0871                        candidate
modification_flag               0.0760  hard_excluded (diagnostic only)
original_ltv                    0.0639                        candidate
original_cltv                   0.0613                        candidate
is_multi_borrower              -0.0467                        

### Check 6 · Row Counts, Schema, NULL Rates, Hard Exclusions, and Training-Window Feature Availability

This section separates two questions:

1. **Should a column ever be eligible to become a model feature?**  
   Some panel columns are retained only for diagnostics / metadata and are hard-excluded regardless of whether they vary.

2. **If a column is eligible in principle, does it vary in the fitted-training window?**  
   Only eligible columns with more than one distinct non-null training value remain in the active feature list.

This distinction matters because several columns are informative for interpretation but not appropriate as primary model inputs:

- `channel` is contaminated by the pre-2008 collection artifact; Freddie Mac did not collect granular broker/correspondent/retail channel information before 2008 (Freddie Mac, 2026a).
- `modification_flag` and `payment_deferral_flag` are affected by policy/reporting transitions and later-regime workout behavior.
- `loan_age`, `remaining_months_to_legal_maturity`, and `amortization_ratio` change semantics after modification because modification can reset loan-age and maturity calculations (Freddie Mac, 2026a, 2026b).
- `deferred_upb_ratio` is effectively a later-regime feature with weak or no historical-training support.
- `in_workout_plan_flag` is unavailable in the training window and intended for later-period diagnostics.

The NULL-rate diagnostics are substantively important because SFLLD sentinel values encode disclosure rules and program-specific masking rather than ordinary random missingness. The missingness flags are retained so later models can learn from missingness patterns without imputing fabricated numeric values (Rubin, 1976; Little & Rubin, 2019; LightGBM Developers, n.d.).

In [ ]:
print("=== Row Count Summary ===")
summary = con.execute("""
SELECT split, COUNT(*) AS n_rows,
       COUNT(DISTINCT loan_sequence_number) AS n_loans,
       ROUND(100.0 * AVG(y), 4) AS pos_rate_pct
FROM panel GROUP BY split ORDER BY MIN(obs_month)
""").df()
print(summary.to_string(index=False))

print("\n=== Schema ===")
schema_df = con.execute("DESCRIBE panel").df()
print(schema_df[["column_name", "column_type"]].to_string(index=False))

schema_map = dict(zip(schema_df["column_name"], schema_df["column_type"]))
valid_int_types = {
    "TINYINT", "SMALLINT", "INTEGER", "BIGINT",
    "UTINYINT", "USMALLINT", "UINTEGER", "UBIGINT"
}
for col in ["obs_year", "vintage_year", "vintage_quarter"]:
    assert schema_map.get(col) in valid_int_types, (
        f"{col} must be integer-coded in panel. Found type={schema_map.get(col)!r}"
    )

print("\n=== NULL rates in selected derived features (training split, %) ===")
null_cols = [
    "upb_rel_change_3m", "amortization_ratio", "rate_spread",
    "deferred_upb_ratio", "current_interest_rate", "original_dti"
]
null_exprs = ", ".join(
    f"ROUND(100.0 * SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) / COUNT(*), 3) AS {c}"
    for c in null_cols
)
null_row = con.execute(f"SELECT {null_exprs} FROM panel WHERE split = 'train'").df()
print(null_row.T.rename(columns={0: 'null_pct_%'}).to_string())

=== Row Count Summary ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

         split    n_rows  n_loans  pos_rate_pct
         train 248091644 12653155        1.1560
   calibration 179344121  9742593        1.1075
 test_subprime 633244002 17677746        2.5252
   test_normal 753732904 18595092        1.5173
    test_covid 201447058 16570540        1.6793
test_rate_hike 306340880 14958763        1.0427

=== Schema ===
                       column_name column_type
              loan_sequence_number     VARCHAR
                         obs_month        DATE
                          obs_year      BIGINT
                                 y     INTEGER
                     is_30dpd_at_t     INTEGER
                          loan_age     INTEGER
remaining_months_to_legal_maturity     INTEGER
                       current_upb      DOUBLE
             current_interest_rate      DOUBLE
                 modification_flag     INTEGER
             payment_deferral_flag     INTEGER
              in_workout_plan_flag     INTEGER
                deferred_upb_ratio   

In [ ]:
# ── Hard exclusions + training-window feature availability ────────────────────
# Only columns not hard-excluded by design are tested for training-window variability.

print("=== Hard feature exclusions (by design) ===")
for f in HARD_EXCLUDED_MODEL_FEATURES:
    print(f"  - {f}")

all_panel_predictor_cols = [
    "is_30dpd_at_t",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "current_upb",
    "current_interest_rate",
    "modification_flag",
    "payment_deferral_flag",
    "in_workout_plan_flag",
    "deferred_upb_ratio",
    "n_times_30dpd_last_12m",
    "max_dlq_last_12m",
    "months_since_last_dlq",
    "upb_rel_change_3m",
    "amortization_ratio",
    "rate_spread",
    "credit_score",
    "fico_missing",
    "original_ltv",
    "ltv_missing",
    "original_cltv",
    "cltv_missing",
    "original_dti",
    "dti_missing",
    "dti_missing_relief_refi",
    "dti_missing_non_relief_refi",
    "original_interest_rate",
    "original_upb",
    "original_loan_term",
    "loan_purpose",
    "channel",
    "occupancy_status",
    "property_type",
    "number_of_units",
    "is_multi_borrower",
    "first_time_homebuyer_flag",
    "vintage_year",
    "vintage_quarter",
    "census_division",
]

candidate_feature_cols = [
    f for f in all_panel_predictor_cols
    if f not in HARD_EXCLUDED_MODEL_FEATURES
]

diagnostic_only_cols = [
    f for f in all_panel_predictor_cols
    if f in HARD_EXCLUDED_MODEL_FEATURES
]

print(f"\nCandidate features eligible for model selection: {len(candidate_feature_cols)}")
print(f"Diagnostic-only / hard-excluded columns        : {len(diagnostic_only_cols)}")

availability_rows = []
for col in candidate_feature_cols:
    q = con.execute(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(*) FILTER (WHERE {col} IS NOT NULL) AS n_non_null,
           COUNT(DISTINCT {col}) FILTER (WHERE {col} IS NOT NULL) AS n_distinct_non_null
    FROM panel
    WHERE split = 'train'
    """).fetchone()

    availability_rows.append({
        "feature": col,
        "n_rows": int(q[0]),
        "n_non_null": int(q[1]),
        "n_distinct_non_null": int(q[2] or 0),
    })

training_feature_availability = (
    pd.DataFrame(availability_rows)
    .sort_values(["n_distinct_non_null", "n_non_null", "feature"], ascending=[True, True, True])
    .reset_index(drop=True)
)

print("\n=== Training-window feature availability (eligible features only) ===")
print(training_feature_availability.to_string(index=False))

training_constant_features = training_feature_availability.loc[
    training_feature_availability["n_distinct_non_null"] <= 1, "feature"
].tolist()

active_feature_cols = training_feature_availability.loc[
    training_feature_availability["n_distinct_non_null"] > 1, "feature"
].tolist()

print()
if training_constant_features:
    print("Eligible features excluded because they are constant/unavailable in training:")
    for f in training_constant_features:
        print(f"  - {f}")
else:
    print("✓  No training-constant eligible features detected.")

print("\nHard-excluded columns retained in the panel for diagnostics / metadata:")
for f in diagnostic_only_cols:
    print(f"  - {f}")

print(f"\nActive model feature count            : {len(active_feature_cols)}")
print(f"Eligible candidate feature count      : {len(candidate_feature_cols)}")
print(f"Hard-excluded diagnostic column count : {len(diagnostic_only_cols)}")

if "rate_spread" in active_feature_cols:
    rs_nonzero_pct = con.execute("""
        SELECT ROUND(100.0 * AVG(CASE WHEN rate_spread IS NOT NULL
                                       AND ABS(rate_spread) > 1e-12
                                      THEN 1.0 ELSE 0.0 END), 4)
        FROM panel WHERE split = 'train'
    """).fetchone()[0]
    print("\n⚠  rate_spread remains active in training.")
    print(f"  Share of training rows with non-zero rate_spread: {rs_nonzero_pct:.4f}%")
    print("  For fixed-rate loans the spread is structurally 0 unless rate-modification activity")
    print("  creates variation. If the non-zero share above is near zero, the feature carries")
    print("  little training-window signal; NB04's SHAP dead-feature check verifies whether the")
    print("  fitted model actually uses it.")
else:
    print("\n✓  rate_spread is not active after the training-window variability screen.")

print("\n=== Vintage support outside training split ===")
train_vintage_years = con.execute("""
    SELECT DISTINCT vintage_year
    FROM panel
    WHERE split = 'train'
""").df()["vintage_year"].dropna().astype(int).tolist()
train_vintage_years = set(train_vintage_years)

for split_name in ["calibration", "test_subprime", "test_normal", "test_covid", "test_rate_hike"]:
    q = con.execute(f"""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(*) FILTER (WHERE vintage_year NOT IN ({",".join(map(str, sorted(train_vintage_years)))})
                         OR vintage_year IS NULL) AS n_rows_unseen_vintage
    FROM panel
    WHERE split = '{split_name}'
    """).fetchone()
    pct = (100.0 * q[1] / q[0]) if q[0] else 0.0
    print(f"  {split_name:<14} unseen-vintage rows = {q[1]:,} / {q[0]:,} ({pct:.2f}%)")

=== Hard feature exclusions (by design) ===
  - channel
  - modification_flag
  - payment_deferral_flag
  - loan_age
  - remaining_months_to_legal_maturity
  - amortization_ratio
  - deferred_upb_ratio
  - in_workout_plan_flag

Candidate features eligible for model selection: 30
Diagnostic-only / hard-excluded columns        : 8

=== Training-window feature availability (eligible features only) ===
                    feature    n_rows  n_non_null  n_distinct_non_null
    dti_missing_relief_refi 248091644   248091644                    1
               cltv_missing 248091644   248091644                    2
                dti_missing 248091644   248091644                    2
dti_missing_non_relief_refi 248091644   248091644                    2
               fico_missing 248091644   248091644                    2
              is_30dpd_at_t 248091644   248091644                    2
          is_multi_borrower 248091644   248091644                    2
                ltv_missing 24

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  calibration    unseen-vintage rows = 29,258,093 / 179,344,121 (16.31%)
  test_subprime  unseen-vintage rows = 357,323,501 / 633,244,002 (56.43%)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  test_normal    unseen-vintage rows = 675,732,477 / 753,732,904 (89.65%)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  test_covid     unseen-vintage rows = 196,611,041 / 201,447,058 (97.60%)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  test_rate_hike unseen-vintage rows = 302,197,412 / 306,340,880 (98.65%)


### Check 7 · Base-to-Panel Loan Reduction

NB02 separately verifies that no performance loan lacks an origination match, so the `INNER JOIN precomp_orig` is lossless in this release. The distinct-loan difference measured below instead reflects loans whose eligible observations fall only in the excluded buffer periods.


In [ ]:
# Compare distinct loans in the unrestricted 1999-2023 base population with
# the final six-window panel. NB02 establishes zero performance loans without
# origination matches, so this difference reflects the excluded buffer periods.

n_base_loans = con.execute(f"""
    SELECT COUNT(DISTINCT loan_sequence_number)
    FROM sf_performance_by_period
    WHERE zero_balance_code IS NULL
      AND loan_age >= 1
      AND current_loan_delinquency_status IN ('0', '00', '1')
      AND reporting_year BETWEEN 1999 AND 2023
""").fetchone()[0]

n_panel_loans = con.execute("""
    SELECT COUNT(DISTINCT loan_sequence_number) FROM panel
""").fetchone()[0]

n_orig_loans = con.execute("""
    SELECT COUNT(DISTINCT loan_sequence_number) FROM precomp_orig
""").fetchone()[0]

dropped = n_base_loans - n_panel_loans
drop_pct = 100.0 * dropped / n_base_loans if n_base_loans else 0.0

print("=== Orphan Loan Drop (INNER JOIN on precomp_orig) ===")
print(f"  Loans in base population (performance)  : {n_base_loans:>12,}")
print(f"  Distinct loans in precomp_orig          : {n_orig_loans:>12,}")
print(f"  Distinct loans in final panel           : {n_panel_loans:>12,}")
print(f"  Dropped by INNER JOIN                   : {dropped:>12,}  ({drop_pct:.4f}%)")
print()

if drop_pct > 0.1:
    print("⚠  Drop exceeds 0.1% - investigate whether loans are missing from origination file.")
else:
    print(f"✓  Orphan drop is negligible ({drop_pct:.4f}%). INNER JOIN introduces no material data loss.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Orphan Loan Drop (INNER JOIN on precomp_orig) ===
  Loans in base population (performance)  :   46,664,954
  Distinct loans in precomp_orig          :   48,597,637
  Distinct loans in final panel           :   46,648,508
  Dropped by INNER JOIN                   :       16,446  (0.0352%)

✓  Orphan drop is negligible (0.0352%). INNER JOIN introduces no material data loss.


### Check 8 · Duplicate Observation Check

Each `(loan_sequence_number, obs_month)` pair must be unique in the panel.
Duplicates would silently inflate training or calibration data and
indicate a bug in the year-loop partitioning or panel-assembly logic.


In [ ]:
n_total_rows = con.execute("SELECT COUNT(*) FROM panel").fetchone()[0]
n_distinct_pairs = con.execute("""
    SELECT COUNT(*) FROM (
        SELECT DISTINCT loan_sequence_number, obs_month FROM panel
    )
""").fetchone()[0]

n_duplicates = n_total_rows - n_distinct_pairs

print("=== Duplicate (loan_sequence_number, obs_month) Check ===")
print(f"  Total panel rows                        : {n_total_rows:>12,}")
print(f"  Distinct (loan, obs_month) pairs        : {n_distinct_pairs:>12,}")
print(f"  Duplicate rows                          : {n_duplicates:>12,}")
print()

assert n_duplicates == 0, (
    f"{n_duplicates:,} duplicate (loan_sequence_number, obs_month) pairs detected. "
)
print("✓  No duplicates. Each (loan_sequence_number, obs_month) pair is unique.")

# Confirm no buffer rows survived into the final split-partitioned panel
n_buffer = con.execute("""
    SELECT COUNT(*) FROM panel WHERE split = 'buffer'
""").fetchone()[0]
assert n_buffer == 0, (
    f"FATAL: {n_buffer:,} rows with split='buffer' found in final panel. "
    "Buffer rows should have been excluded during §5 split partitioning."
)
print("✓  No buffer rows in final panel (inter-split gap periods correctly excluded).")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Duplicate (loan_sequence_number, obs_month) Check ===
  Total panel rows                        : 2,322,200,609
  Distinct (loan, obs_month) pairs        : 2,322,200,609
  Duplicate rows                          :            0

✓  No duplicates. Each (loan_sequence_number, obs_month) pair is unique.
✓  No buffer rows in final panel (inter-split gap periods correctly excluded).


### Check 9 · Fast-Default Label Gap

The label construction restricts positive events to active performance records where `zero_balance_code IS NULL`. This means that loans terminating via a credit event - especially `ZBC = 02` Third Party Sale or `ZBC = 03` Short Sale / Charge Off - could be missed if they never showed an active 60+ DPD or REO Acquisition status before the terminal record.

This check quantifies the resulting false-negative gap by identifying loans whose credit-event termination record (`ZBC ∈ {02, 03}`) was never preceded by an active 60+ DPD / RA/R status record. These are “fast-default” cases from the perspective of the active-state label construction.

`ZBC = 09` is deliberately excluded from this gap check because the REO path should pass through REO Acquisition status (`RA` or legacy `R`) before REO disposition. `ZBC = 15` and `ZBC = 16` are also excluded because they correspond to whole-loan sale / note sale or reperforming-loan securitization categories rather than the specific short-sale / third-party-sale credit-event gap being bounded here (Freddie Mac, 2026a, 2026b, 2026c).

The result bounds the magnitude of a known label-definition limitation. If material, it is acknowledged in the thesis positive-class definition.

In [ ]:
# Count loans with a credit-event termination (ZBC in 02, 03)
n_credit_event_loans = con.execute(f"""
    SELECT COUNT(DISTINCT loan_sequence_number)
    FROM sf_performance_by_period
    WHERE zero_balance_code IN ('02', '03')
""").fetchone()[0]

# Among those, which never had an active 60+ DPD or RA/R record?
# A loan that *did* pass through 60+ DPD while active is correctly labeled by precomp_label.
# A loan that did NOT is a fast-default case.
n_captured = con.execute(f"""
    SELECT COUNT(DISTINCT loan_sequence_number)
    FROM sf_performance_by_period
    WHERE zero_balance_code IS NULL
      AND (
            TRY_CAST(current_loan_delinquency_status AS INTEGER) >= 2
            OR current_loan_delinquency_status IN ('RA', 'R')
      )
      AND loan_sequence_number IN (
            SELECT DISTINCT loan_sequence_number
            FROM sf_performance_by_period
            WHERE zero_balance_code IN ('02', '03')
      )
""").fetchone()[0]

n_fast_default = n_credit_event_loans - n_captured
fast_default_pct = 100.0 * n_fast_default / n_credit_event_loans if n_credit_event_loans > 0 else 0.0

# What fraction of total positive-class base is the fast-default gap?
# Compare against total loans ever in precomp_label to bound its share.
n_labeled_positive_loans = con.execute(f"""
    SELECT COUNT(DISTINCT loan_sequence_number)
    FROM read_parquet('{label_glob}', union_by_name=True)
""").fetchone()[0]

gap_vs_labeled = 100.0 * n_fast_default / n_labeled_positive_loans if n_labeled_positive_loans > 0 else 0.0

print("=== Fast-Default Label Gap (ZBC=02/03 credit-event terminations) ===")
print(f"  Loans with ZBC ∈ {{02,03}} (credit-event termination) : {n_credit_event_loans:>12,}")
print(f"  Of those: had active 60+ DPD record (captured by label): {n_captured:>12,}")
print(f"  Fast-default loans (no active 60+ DPD record)          : {n_fast_default:>12,}  ({fast_default_pct:.2f}%)")
print(f"  Fast-default count as % of labeled-positive loans      : {gap_vs_labeled:.2f}%")
print()

if fast_default_pct > 5.0:
    print("⚠  Fast-default gap exceeds 5% of credit-event terminations. Requires explicit")
    print("   discussion in the thesis positive-class definition section.")
elif fast_default_pct > 1.0:
    print("⚠  Fast-default gap is non-negligible.")
else:
    print(f"✓  Fast-default gap is small ({fast_default_pct:.2f}% of ZBC=02/03 loans). Label completeness is adequate.")

print()
print("Note: this gap is inherent to the active-state-only label construction design.")
print("      The transition-risk model scope (current/30-DPD base population) means the")
print("      affected cases are loans that terminated via selected credit-event ZBCs before")
print("      the normal 60+ DPD / REO Acquisition path was visible in active records.")
print("      The above count bounds this label-definition limitation for thesis discussion.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Fast-Default Label Gap (ZBC=02/03 credit-event terminations) ===
  Loans with ZBC ∈ {02,03} (credit-event termination) :      240,233
  Of those: had active 60+ DPD record (captured by label):      235,856
  Fast-default loans (no active 60+ DPD record)          :        4,377  (1.82%)
  Fast-default count as % of labeled-positive loans      : 0.17%

⚠  Fast-default gap is non-negligible.

Note: this gap is inherent to the active-state-only label construction design.
      The transition-risk model scope (current/30-DPD base population) means the
      affected cases are loans that terminated via selected credit-event ZBCs before
      the normal 60+ DPD / REO Acquisition path was visible in active records.
      The above count bounds this label-definition limitation for thesis discussion.


### Check 10 · Within-Split Positive Rate Trend

Checks whether positive rates are stable or systematically trending within individual test splits. Intra-split non-stationarity matters for CP evaluation because standard conformal validity relies on exchangeability between calibration and test observations (Vovk et al., 2005; Shafer & Vovk, 2008).

1. CP coverage is calibrated using a single calibration score distribution. If the test split itself drifts over time, then the relevant nonconformity-score distribution is changing even within that test split.
2. Comparing aggregate coverage statistics across splits, such as `test_subprime` versus `test_normal`, implicitly compresses each split into one summary regime. Strong intra-split trends weaken that simplification and should be noted when interpreting NB06/NB07 results.
3. Time-based splits are appropriate for temporal dependence, but they do not eliminate regime drift within a split (Roberts et al., 2017).

The check computes annual positive rates within each split and reports the trend direction. It does not assert pass/fail; it produces evidence for the thesis discussion of regime stability and exchangeability violations.


In [ ]:
print("=== Within-Split Positive Rate Trend (annual) ===")
print()

for split_name in ["calibration", "test_subprime", "test_normal", "test_covid", "test_rate_hike"]:
    annual = con.execute(f"""
        SELECT obs_year,
               COUNT(*) AS n_rows,
               ROUND(100.0 * AVG(y), 4) AS pos_rate_pct
        FROM panel
        WHERE split = '{split_name}'
        GROUP BY obs_year
        ORDER BY obs_year
    """).df()

    if len(annual) < 2:
        print(f"  {split_name:<18}: only 1 year - trend not computable")
        continue

    rates = annual["pos_rate_pct"].tolist()
    diffs = [rates[i+1] - rates[i] for i in range(len(rates)-1)]
    monotone_up   = all(d > 0 for d in diffs)
    monotone_down = all(d < 0 for d in diffs)
    direction = "↑ monotone increasing" if monotone_up else ("↓ monotone decreasing" if monotone_down else "↕ non-monotone")

    max_swing = max(rates) - min(rates)
    print(f"  {split_name:<18}: {direction}  (range {min(rates):.3f}-{max(rates):.3f}%,  swing={max_swing:.3f} pp)")
    for _, row in annual.iterrows():
        print(f"    {int(row['obs_year'])}: {row['pos_rate_pct']:.4f}%  ({int(row['n_rows']):,} rows)")
    print()

print("Note: strong intra-split monotone trends indicate the split contains multiple sub-regimes.")

=== Within-Split Positive Rate Trend (annual) ===



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  calibration       : ↓ monotone decreasing  (range 1.090–1.126%,  swing=0.035 pp)
    2005: 1.1258%  (86,482,468 rows)
    2006: 1.0904%  (92,861,653 rows)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  test_subprime     : ↕ non-monotone  (range 1.554–3.490%,  swing=1.935 pp)
    2007: 1.5545%  (98,051,836 rows)
    2008: 2.9118%  (104,261,511 rows)
    2009: 3.4897%  (106,881,627 rows)
    2010: 2.6665%  (110,025,719 rows)
    2011: 2.3858%  (109,504,404 rows)
    2012: 2.0609%  (104,518,905 rows)



...

  test_normal       : ↕ non-monotone  (range 1.101–2.748%,  swing=1.647 pp)
    2013: 1.6993%  (104,135,124 rows)
    2014: 1.4542%  (106,689,556 rows)
    2015: 1.2737%  (109,147,380 rows)
    2016: 1.2484%  (111,640,517 rows)
    2017: 1.3640%  (114,767,136 rows)
    2018: 1.1006%  (117,238,740 rows)
    2019: 2.7476%  (90,114,451 rows)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  test_covid        : ↓ monotone decreasing  (range 0.952–2.401%,  swing=1.449 pp)
    2020: 2.4005%  (101,152,630 rows)
    2021: 0.9519%  (100,294,428 rows)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  test_rate_hike    : ↑ monotone increasing  (range 0.991–1.093%,  swing=0.101 pp)
    2022: 0.9914%  (151,205,691 rows)
    2023: 1.0926%  (155,135,189 rows)

Note: strong intra-split monotone trends indicate the split contains multiple sub-regimes.


---
## Section 6 · Output Manifest

`panel_manifest.json` is the primary machine-readable contract exported by this notebook for downstream stages.

Records:
- panel location, split windows, alpha, target column
- active model features (eligible + variable in training)
- hard-excluded model features retained only for diagnostics / metadata
- training-window feature availability diagnostics
- split-level class balance and `scale_pos_weight_estimate`
- calibration subgroup sizes for Mondrian CP
- SFLLD release metadata inherited from the ingestion manifest
- sentinel / NULL-preservation conventions that downstream notebooks must respect

The `scale_pos_weight_estimate` is stored as a descriptive class-imbalance diagnostic for the full declared training split; NB04 recomputes the fitted-model weight from the actual 1999-2003 fitting rows.

**Critical conventions for NB04 and the thesis write-up:**

1. `target_label` is **"60+ DPD or REO Acquisition"**. This follows the combination of numeric delinquency buckets and REO Acquisition status codes in the SFLLD monthly-performance data (Freddie Mac, 2026b, 2026c).
2. `test_normal` ends at **2019-09-30** to reduce, but not eliminate, COVID forward-window contamination.
3. `test_covid` should be interpreted as both a macro regime and a policy/reporting regime because CARES-era forbearance affected mortgage-servicing behavior (CFPB & CSBS, 2020; FHFA Office of Inspector General, 2020).
4. `credit_score`, `original_ltv`, `original_cltv`, and `original_dti` preserve `NULL` when not observed under SFLLD disclosure rules; do not replace these with arbitrary numeric imputations (Freddie Mac, 2026a, 2026b; Rubin, 1976; Little & Rubin, 2019).
5. `is_multi_borrower` is the binary transform used to harmonise the 2018Q2    borrower-count cardinality change; see the §3.1 frozen-artifact note.
6. `channel` is retained in the panel but is hard-excluded from active model features because pre-2008 channel values are not granularly collected (Freddie Mac, 2026a).
7. Modified-loan timing variables and policy-transition variables are retained in the panel but excluded from the main model feature set.
8. `upb_rel_change_3m` is suppressed for modified observations so it cannot proxy for excluded modification-period information.
9. Active model features must respect the prediction-time information boundary: features may use information available at or before observation month `t`, but must not encode information from the forward label window `(t, t+12]` (Kaufman et al., 2012).
10. NB04 fits on `obs_year < 2004` and validates on `obs_year == 2004`; the effective fitted-training window is 1999-2003, not the full declared `split='train'`.
11. `vintage_year` and `vintage_quarter` must remain integer-coded in the final panel schema.
12. Later CP results must be interpreted as empirical stress tests under panel dependence and regime shift, not as evidence that exact exchangeability holds in the SFLLD panel (Vovk et al., 2005; Shafer & Vovk, 2008; Wooldridge, 2010).

In [ ]:
manifest_path = MANIFEST_DIR / "panel_manifest.json"

_manifest_exists = False
if manifest_path.exists() and not FORCE_REBUILD_MANIFEST:
    try:
        with open(manifest_path) as _f:
            _existing = json.load(_f)
        if (
            _existing.get("feature_cols")
            and _existing.get("split_statistics")
            and _existing.get("hard_excluded_model_feature_cols") is not None
        ):
            print("✓  panel_manifest.json already exists - skipping rebuild.")
            print(f"   Created   : {_existing.get('created_at','?')}")
            print(f"   Features  : {len(_existing['feature_cols'])}  |  Subgroups: {len(_existing.get('subgroup_cols',[]))}")
            manifest = _existing
            _manifest_exists = True
    except (json.JSONDecodeError, KeyError):
        print("⚠  panel_manifest.json invalid - rebuilding.")

if FORCE_REBUILD_MANIFEST and manifest_path.exists():
    manifest_path.unlink()
    print("⚠  FORCE_REBUILD_MANIFEST=True - deleted cached panel_manifest.json so it can be rebuilt.")

if not _manifest_exists:
    split_stats = con.execute("""
    SELECT split, COUNT(*) AS n_rows,
           COUNT(DISTINCT loan_sequence_number) AS n_distinct_loans,
           CAST(SUM(y) AS BIGINT) AS n_positive,
           ROUND(100.0 * AVG(y), 6) AS positive_rate_pct,
           MIN(obs_month)::VARCHAR AS obs_start,
           MAX(obs_month)::VARCHAR AS obs_end
    FROM panel
    GROUP BY split
    ORDER BY obs_start
    """).df()

    train_row = split_stats[split_stats["split"] == "train"].iloc[0]
    n_train_pos = int(train_row["n_positive"])
    n_train_neg = int(train_row["n_rows"]) - n_train_pos
    spw_estimate = round(n_train_neg / n_train_pos, 2) if n_train_pos > 0 else None

    subgroup_cols = ["fico_tier", "ltv_bucket", "fico_ltv_group", "census_division"]

    meta_cols = [
        # Panel partitioning column
        "split",
        # Disclosure-regime and loan-program identifiers (never active model features;
        # origination_rule_group is a CRT-eligibility classifier, not a credit-risk signal)
        "origination_rule_group",
        "property_state",
        "harp_flag",
        "relief_refi_flag",
        # Hard-excluded dynamic performance columns retained for later-regime diagnostics
        "in_workout_plan_flag",
        "modification_flag",
        "payment_deferral_flag",
        "loan_age",
        "remaining_months_to_legal_maturity",
        "amortization_ratio",
        "deferred_upb_ratio",
        # Channel: hard-excluded model feature retained for origination-era diagnostics
        "channel",
    ]
    _id_cols = {"loan_sequence_number", "obs_month", "obs_year"}
    assert not (set(meta_cols) & set(active_feature_cols)), (
        f"meta_cols overlaps active features: {sorted(set(meta_cols) & set(active_feature_cols))}"
    )
    assert not (set(meta_cols) & _id_cols), (
        f"meta_cols overlaps id columns: {sorted(set(meta_cols) & _id_cols)}"
    )

    manifest = {
        "created_at": datetime.now(tz=timezone.utc).isoformat(),
        "alpha": ALPHA,
        "target_col": "y",
        "subgroup_cols": subgroup_cols,
        "meta_cols": meta_cols,
        "candidate_feature_cols": candidate_feature_cols,
        "hard_excluded_model_feature_cols": HARD_EXCLUDED_MODEL_FEATURES,
        "feature_cols": active_feature_cols,
        "excluded_training_constant_feature_cols": training_constant_features,
        "split_windows": SPLIT_WINDOWS,
        "split_statistics": split_stats.to_dict(orient="records"),
        "forward_window_contamination": {
            "calibration" : CALIBRATION_CONTAM,
            "test_normal" : TEST_NORMAL_COVID_CONTAM,
        },
        "scale_pos_weight_estimate": spw_estimate,
        "target_label": "60+ DPD or REO Acquisition",
    }

    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2, default=str)

    print(f"✓  panel_manifest.json written: {manifest_path}")
    print(f"   Active model features       : {len(manifest['feature_cols'])}")
    print(f"   Eligible candidate count    : {len(manifest['candidate_feature_cols'])}")
    print(f"   Hard exclusions             : {len(manifest['hard_excluded_model_feature_cols'])}")
    print(f"   Excluded constants          : {len(manifest['excluded_training_constant_feature_cols'])}")
    print(f"   scale_pos_weight estimate   : {spw_estimate}")

print("\n" + "=" * 60)
print("PANEL CONSTRUCTION COMPLETE")
print("=" * 60)
for row in manifest["split_statistics"]:
    print(
        f"{row['split']:<14} "
        f"rows={row['n_rows']:,}  "
        f"loans={row['n_distinct_loans']:,}  "
        f"pos={row['n_positive']:,}  "
        f"rate={row['positive_rate_pct']:.4f}%  "
        f"{row['obs_start']} → {row['obs_end']}"
    )

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓  panel_manifest.json written: /content/drive/MyDrive/master_thesis/manifests/panel_manifest.json
   Active model features       : 29
   Eligible candidate count    : 30
   Hard exclusions             : 8
   Excluded constants          : 1
   scale_pos_weight estimate   : 85.51

PANEL CONSTRUCTION COMPLETE
train          rows=248,091,644  loans=12,653,155  pos=2,867,880  rate=1.1560%  1999-01-01 → 2004-12-01
calibration    rows=179,344,121  loans=9,742,593  pos=1,986,205  rate=1.1075%  2005-01-01 → 2006-12-01
test_subprime  rows=633,244,002  loans=17,677,746  pos=15,990,464  rate=2.5252%  2007-01-01 → 2012-12-01
test_normal    rows=753,732,904  loans=18,595,092  pos=11,436,593  rate=1.5173%  2013-01-01 → 2019-09-01
test_covid     rows=201,447,058  loans=16,570,540  pos=3,382,830  rate=1.6793%  2020-03-01 → 2021-09-01
test_rate_hike rows=306,340,880  loans=14,958,763  pos=3,194,065  rate=1.0427%  2022-01-01 → 2023-12-01


---
## Section 7 · Close Connection

In [ ]:
con.close()
print("DuckDB connection closed.")

DuckDB connection closed.


---

## Appendix · References

### Freddie Mac documentation

- Freddie Mac. (2026a, January). Single-family loan-level dataset frequently asked questions (FAQ) (Frequently Asked Questions). Freddie Mac. Retrieved March 3, 2026, from https://www.freddiemac.com/research/datasets/sf-loanlevel-dataset

- Freddie Mac. (2026b, January). Single-family loan-level dataset general user guide (User Guide). Freddie Mac. Retrieved March 3, 2026, from https://www.freddiemac.com/research/datasets/sf-loanlevel-dataset

- Freddie Mac. (2026c, January). Single-family loan-level dataset release notes (Release Notes). Freddie Mac. Retrieved March 3, 2026, from https://www.freddiemac.com/research/datasets/sf-loanlevel-dataset

### Institutional and policy sources

- Conference of State Bank Supervisors. (2020, June 4). CARES act forbearance & foreclosure. https://www.csbs.org/cares-act-forbearance-foreclosure

- Federal Housing Finance Agency Office of Inspector General. (2020, July 27). Oversight by fannie mae and freddie mac of compliance with forbearance requirements under the CARES Act and implementing guidance by mortgage servicers (Special Report No. OIG-2020-004). https://www.fhfaoig.gov/sites/default/files/OIG-2020-004.pdf

- U.S. Census Bureau. (2010). Census regions and divisions of the united states. https://www2.census.gov/geo/pdfs/maps-data/maps/reference/us_regdiv.pdf

### Mortgage-risk, competing-risk, and credit-risk modeling literature

- Campbell, J. Y., & Cocco, J. F. (2015). A model of mortgage default. The Journal of Finance, 70(4), 1495–1554. https://doi.org/10.1111/jofi.12252

- Deng, Y., Quigley, J. M., & Van Order, R. (2000). Mortgage terminations, heterogeneity and the exercise of mortgage options. Econometrica, 68(2), 275–307. https://doi.org/10.1111/1468-0262.00110

- Fitzpatrick, T., & Mues, C. (2016). An empirical comparison of classification algorithms for mortgage default prediction: Evidence from a distressed mortgage market. European Journal of Operational Research, 249(2), 427–439. https://doi.org/10.1016/j.ejor.2015.09.014

- Foote, C. L., Gerardi, K., & Willen, P. S. (2008). Negative equity and foreclosure: Theory and evidence. Journal of Urban Economics, 64(2), 234–245. https://doi.org/10.1016/j.jue.2008.07.006

- Lessmann, S., Baesens, B., Seow, H.-V., & Thomas, L. C. (2015). Benchmarking state-of-the-art classification algorithms for credit scoring: An update of research. European Journal of Operational Research, 247(1), 124–136. https://doi.org/10.1016/j.ejor.2015.05.030

- Shumway, T. (2001). Forecasting bankruptcy more accurately: A simple hazard model. The Journal of Business, 74(1), 101–124. https://doi.org/10.1086/209665

### Conformal prediction, validation design, panel data, event-history, and missing-data literature

- Kaufman, S., Rosset, S., Perlich, C., & Stitelman, O. (2012). Leakage in data mining: Formulation, detection, and avoidance. ACM Transactions on Knowledge Discovery from Data, 6(4), Article 15, 1–21. https://doi.org/10.1145/2382577.2382579

- Ke, G., Meng, Q., Finley, T., Wang, T., Chen, W., Ma, W., Ye, Q., & Liu, T.-Y. (2017). LightGBM: A highly efficient gradient boosting decision tree. Advances in Neural Information Processing Systems, 30, 3146–3154. https://proceedings.neurips.cc/paper/2017/hash/6449f44a102fde848669bdd9eb6b7Abstract.html

- Kuhn, M., & Johnson, K. (2013). Applied predictive modeling. Springer. https://doi.org/10.1007/978-1-4614-6849-3

- LightGBM Developers. (n.d.). *Advanced topics: Missing value handle.* LightGBM documentation. https://lightgbm.readthedocs.io/en/latest/Advanced-Topics.html

- Little, R. J. A., & Rubin, D. B. (2019). Statistical analysis with missing data (3rd ed.). Wiley. https://doi.org/10.1002/9781119482260

- Roberts, D. R., Bahn, V., Ciuti, S., Boyce, M. S., Elith, J., Guillera-Arroita, G., Hauenstein, S., Lahoz-Monfort, J. J., Schröder, B., Thuiller, W., Warton, D. I., Wintle, B. A., Hartig, F., & Dormann, C. F. (2017). Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure. Ecography, 40(8), 913–929. https://doi.org/10.1111/ecog.02881

- Rubin, D. B. (1976). Inference and missing data. Biometrika, 63(3), 581–592. https://doi.org/10.1093/biomet/63.3.581

- Shafer, G., & Vovk, V. (2008). A tutorial on conformal prediction. Journal of Machine Learning Re-
search, 9(12), 371–421. https://www.jmlr.org/papers/v9/shafer08a.html

- Singer, J. D., & Willett, J. B. (1993). It’s about time: Using discrete-time survival analysis to study duration and the timing of events. Journal of Educational Statistics, 18(2), 155–195. https://doi.org/10.3102/10769986018002155

- Vovk, V. (2012). Conditional validity of inductive conformal predictors. In S. C. H. Hoi & W. Buntine (Eds.), Proceedings of the asian conference on machine learning (pp. 475–490, Vol. 25). PMLR. https://proceedings.mlr.press/v25/vovk12.html

- Vovk, V., Gammerman, A., & Shafer, G. (2005). Algorithmic learning in a random world (1st ed.). Springer. https://doi.org/10.1007/b106715

- Wooldridge, J. M. (2010). *Econometric analysis of cross section and panel data* (2nd ed.). MIT Press.